# Football Tracking -- notebook 1 of 2: tracking + zero-switch relinking (Deep-EIoU)
Produces pure player pieces (no ID switches), goalkeepers forced to IDs 1-2 and referees to IDs 3-5. Grouping the pieces
into named players is done in **notebook 2** (`notebook2_player_grouping_naming.ipynb`).

**Outputs for notebook 2** (put them in a dataset):
* `barca_atletico_tracking_cache_deep_eiou_relinked_forced.pkl` (Cell 14)
* `grouping_inputs.pkl` (Cell 14b: per-piece appearance, time segments, pitch anchors, team model)
* `shot_boundaries.npz` (Cell 7b cache -- makes re-runs fast)

**Reproducibility:** your reviewed `union_groups.csv` refers to piece IDs of *this exact run*. Re-running with the same
inputs and settings gives the same IDs (seeded, deterministic). Changing any setting renumbers the pieces and the
review no longer matches (notebook 2 checks this and warns).


Same global re-linking as the BoT-SORT version (sparse greedy linking, pitch-space motion,
per-class max gaps, ghost removal, ratio-aware class locking), adapted to the **Deep-EIoU**
output (`deep_eiou_tracks.parquet` + `deep_eiou_track_embeddings.npz`, RF-DETR detections).

Changes from the BoT-SORT version:

1. **Input = Deep-EIoU tracks.** No BoT-SORT pass, no boxmot/torchreid install. The per-frame class
   is the raw RF-DETR class of the matched detection (no smoothed-label problem).
2. **Embeddings come from the `.npz` (every 5th frame).** Head/tail appearance uses the first/last
   `EMBED_SAMPLES` stored embeddings (~2 s) instead of the first/last 8 frames.
3. **Selective splitting instead of splitting at every contact.** Deep-EIoU tracks are mostly clean,
   so each track is cut only where there is evidence of a problem: a sustained class switch, an abrupt
   box jump (ID swap or camera cut), or a sharp appearance drift. `SPLIT_MODE = "all_contacts"`
   restores the old behaviour for comparison.
4. **Deep-EIoU tracks are not split at frame gaps.** A Deep-EIoU track that resumes after a short
   gap (<= 2 s) is a deliberate re-association, not a new track.
5. **The manual track-2 / track-41 splits are removed.** Those IDs belonged to the BoT-SORT run.
   They are replaced by an automatic sustained-class-switch split on the final IDs (Cell 12b), which
   finds the same kind of contamination without hard-coded track numbers or frames.
6. **The Cell 13b check covers every class**, using the stored raw classes (no IoU re-matching).
7. **Final audit (Cell 15)** runs the same label-free metrics as the tracker comparison, on the raw
   Deep-EIoU tracks and on the relinked output, side by side.

Priority is unchanged: zero ID switches over the exact ground-truth count. Linking parameters are the
BoT-SORT-era values -- validate them on this input before tuning.

### v2 -- minimum IDs, goalkeepers forced to 2 IDs, referees forced to 3 IDs (players not forced)

8. **Bug fix: head/tail appearance vectors are re-normalised.** The mean of L2-normalised embeddings has
   norm < 1, so `np.dot(tail_emb, head_emb)` was systematically *below* the true cosine and many correct
   links fell under `MIN_LINK_SCORE`. Every mean embedding is now re-normalised.
9. **Physical-feasibility gate on every link.** A link (or merge) is rejected when the pitch distance between
   the last-seen and first-seen positions is larger than `MAX_SPEED_MPS * gap + POS_SLACK_M`. This only removes
   impossible links, so the appearance side can be loosened without adding ID switches.
10. **Goalkeepers -> exactly 2 fixed IDs (1 = left half, 2 = right half), referees -> exactly 3 fixed IDs
    (3 = main referee, 4 = assistant on touchline y~0, 5 = assistant on touchline y~68).** GK/referee tracklets are
    no longer chained by appearance; each one is assigned to a slot by pitch position (+ an appearance gate), then
    a per-slot occupancy check guarantees one box per slot per frame. Tracklets that cannot fit are dropped (logged).
11. **Player identity-level merge (Cell 10b).** After short-gap chaining, whole player identities are merged
    agglomeratively with no gap limit, under hard constraints: never co-visible, same team (2-means on embeddings),
    physically reachable at every boundary. Threshold sweep printed first; players are not forced to 20.
12. **Class locking:** forced IDs keep their slot class, every other ID is locked to `player` (a player ID can no
    longer become an extra goalkeeper/referee). The final-ID class-switch split (Cell 12b) is report-only by default.
13. **Strict stage 1, permissive-but-constrained stage 2.** `MIN_LINK_SCORE` 0.3 -> 0.5 (the score scale changed with
    fix 8). A wrong short-gap link produces a mixed identity that is co-visible with both people, so Cell 10b can never
    merge it away; leaving a correct link for Cell 10b costs nothing.
14. **Pitch-space teleport cut (Cell 8, `pitch_jump`).** Some Deep-EIoU tracks stay alive for 10+ minutes by
    hopping between players (mostly across camera cuts); same kit, so the drift and image-jump checks miss it. A track
    is cut where the median pitch position of the 5 observations before vs after moves faster than
    `PITCH_JUMP_SPEED_MPS` (+ slack). The feasibility gate then stops linking/merging from rejoining the pieces.
15. **Cell 16 -- contact sheet per final ID** (one crop per stitched piece + one every 30 s) and **Cell 17 -- trace
    an ID switch** (how each piece of an ID was joined: raw track, stage-1 link, 10b merge, slot).
16. **One box per (frame, ID)** is enforced when the cache is rebuilt (small handoff overlaps are trimmed).


### v4 -- ZERO player ID switches (priority over the number of IDs)
Goal: every player ID must contain one person only, so IDs can be unioned manually afterwards. Fragmentation is
accepted; an ID switch is not. With `STRICT_NO_SWITCH = True` (Cell 1):

17. **Cut at every camera cut (new Cell 7b).** Shot boundaries are detected once from the video (HSV-histogram
    difference, cached to disk). Deep-EIoU keeps IDs alive across cuts and hands them to whoever is in that part of
    the screen -- the main source of the 12-minute mixed tracks. Every track is cut at every shot boundary, and no
    link is allowed across one (`LINK_ACROSS_SHOTS = False`).
18. **Cut around every player contact, and drop the contact frames.** Tracker switches happen when two boxes touch.
    Player tracks are cut before and after every contact (boxes enlarged by `CONTACT_EXPAND`), and the ambiguous
    contact piece itself is dropped (`DROP_CONTACT_SEGMENTS`).
19. **Only unambiguous stage-1 links.** A link needs: same shot, gap <= 2 s, valid pitch positions on both sides,
    score >= 0.6, and it must be the mutual best choice by at least `LINK_MIN_MARGIN` over the runner-up on both
    sides. Two teammates emerging from a contact look alike -> small margin -> no link (you union them later).
20. **No identity-level merge** (`PLAYER_MERGE = False`) and a stricter drift cut (`DRIFT_SPLIT_THRESH = 0.2`).
    Goalkeepers / referees keep the forced 2 / 3 slots (already clean).


### v5 -- fewer pieces (still zero switches) + union assistant
21. **Team-aware contacts.** Every embedding is labelled with a team (2-means on player embeddings, new Cell 7c).
    Only contacts between two players of the **same team** are cut; a swap with an opponent changes the team label and
    is cut exactly where it happens by the new `team_switch` cut. Opponent duels no longer shred tracks.
22. **Links across a camera cut only when position proves it**: gap <= `CROSS_SHOT_MAX_GAP` frames, pitch distance
    <= `CROSS_SHOT_MAX_DIST_M`, plus the margin rule. Links never cross teams.
23. **Cell 18 -- union assistant.** Proposes one group per player (hard constraints: never co-visible, same team,
    reachable) and draws one sheet per group, least-similar tiles first, plus a CSV. You only correct the proposals.
    **Cell 19** applies the CSV and refuses any union of co-visible IDs.


## Cell 1 -- Config

In [ ]:
import os
import cv2
import json
import pickle
import bisect
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

# ---- inputs ----
DEEP_EIOU_TRACKS_PATH = "/kaggle/input/deep-eiou-full-match/deep_eiou_tracks.parquet"            # set to your dataset
DEEP_EIOU_EMBS_PATH   = "/kaggle/input/deep-eiou-full-match/deep_eiou_track_embeddings.npz"
BALL_DETECTION_CACHE_PATH = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_firsthalf_detection_cache_ft3.pkl"  # ball only (ball model unchanged)
HOMOGRAPHY_PATH = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/homography_cache_barca_atletico_firsthalf.pkl"
VIDEO_PATH      = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv"

# ---- outputs (new name, so the BoT-SORT-based cache is not overwritten) ----
OUTPUT_TRACKING_CACHE_PATH = "/kaggle/working/barca_atletico_tracking_cache_deep_eiou_relinked_forced.pkl"
OUTPUT_SUMMARY_PATH        = "/kaggle/working/relink_id_summary_forced.csv"
OUTPUT_AUDIT_PATH          = "/kaggle/working/relink_audit_forced.csv"

# Class ids used in the Deep-EIoU run (names were stored as strings in the RF-DETR cache)
CLS_ID_TO_NAME = {1: "player", 2: "referee", 3: "goalkeeper"}
CLASS_TO_ID = {"player": 0, "goalkeeper": 1, "referee": 2}   # kept for compatibility with downstream code

FPS = 25
PX_PER_METER = 10
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

# ---- splitting ----
SPLIT_MODE = "selective"          # "selective" | "strict" (v4: selective + shots + contacts) | "all_contacts" | "none"
CLASS_SWITCH_SMOOTH_FRAMES = 25   # 1 s majority smoothing of the per-frame class
CLASS_SWITCH_MIN_RUN = 50         # a class run must last >= 2 s to count as a real switch (shorter = flicker)
JUMP_MAX_GAP = 3                  # frames; jumps are only checked between near-consecutive observations
JUMP_RATIO = 0.75                 # centre displacement / box height
DRIFT_WINDOW_FRAMES = 50          # 2 s windows
DRIFT_SPLIT_THRESH = 0.3          # cosine distance between adjacent windows
CONTACT_IOU_THRESH = 0.3          # only used when SPLIT_MODE == "all_contacts"

# ---- pre-link noise filtering ----
MIN_TRACKLET_LEN = 20

# ---- global linking parameters (BoT-SORT-era values) ----
MAX_LINK_GAP_BY_CLASS = {"player": 500, "goalkeeper": 70326, "referee": 70326}
VELOCITY_EXTRAPOLATION_CAP_FRAMES = 25     # was 250: an 8-frame velocity extrapolated 10 s is noise
MIN_LINK_SCORE = 0.5                # was 0.3: sims are now true cosines (higher), and a wrong stage-1 link
                                    # creates a mixed ID that also BLOCKS later merges (co-visibility) -> stay strict here
EMBED_WINDOW = 8                  # frames, for velocity fits
EMBED_SAMPLES = 10                # stored embeddings used for head/tail appearance (~2 s at stride 5)
MOTION_WEIGHT = 0.3
MOTION_NORM_M = 8.0
MOTION_NORM_PX = 300.0

# ---- final ghost-track filter + ratio-aware class locking ----
MIN_TRACK_LENGTH = 300
MIN_CONFIRM_FRAMES_ABS = 10
MIN_CONFIRM_RATIO = 0.02
MAX_IDS_PER_CLASS_EXPECTED = {"goalkeeper": 2, "referee": 3}   # sanity check only, not enforced

DEBUG_LINKING = False

# ---- v3: pitch-space teleport cut inside raw tracks (Cell 8) ----
PITCH_JUMP_SPEED_MPS = 12.0       # faster than any sprint
PITCH_JUMP_SLACK_M = 3.0          # homography noise between the two medians
PITCH_JUMP_CTX = 5                # observations on each side (median -> robust to 1-frame homography glitches)

# ---- v2: physical feasibility gate (stage-1 links AND identity merges) ----
MAX_SPEED_MPS = 10.0              # upper bound on sprint speed
POS_SLACK_M = 4.0                 # homography error + foot-anchor noise

# ---- v2: forced officials (GK -> 2 IDs, referee -> 3 IDs) ----
FORCE_OFFICIALS = True
GK_SLOT_IDS  = {"GK_left": 1, "GK_right": 2}                  # pitch x < 52.5 m -> left (first half: no side swap)
REF_SLOT_IDS = {"REF_main": 3, "AR_y0": 4, "AR_y68": 5}       # assistants patrol one touchline each
FIRST_PLAYER_ID = 6
GK_ASSIGN_MODE = "side"           # "side" (pitch half) | "appearance" (2-means on GK embeddings; use if homography sides are unreliable)
POS_SAMPLE_STRIDE = 5             # frames between sampled pitch positions for slot assignment
AR_BAND_M = 7.0                   # an assistant lives within this distance of his touchline
AR_MIN_FRAC = 0.6                 # share of a tracklet's positions inside the band to call it an assistant
AR_ENFORCE_HALF = True            # an assistant patrols one half: reject AR candidates clearly in the other half
AR_HALF_MARGIN_M = 3.0
OFFICIAL_MIN_SIM = 0.5            # cosine to the slot prototype; below -> not this official -> dropped
SLOT_OVERLAP_TOL = 5              # frames a tracklet may overlap its slot (overlapping frames are trimmed)

# ---- v2: player identity-level merge (players are NOT forced to 20) ----
PLAYER_MERGE = True
PLAYER_MERGE_SIM = 0.75           # pick from the sweep printed in Cell 10b
PLAYER_MERGE_SWEEP = [0.85, 0.80, 0.75, 0.70, 0.65, 0.60]
MERGE_OVERLAP_TOL = 5             # co-visible frames tolerated between two identities (handoff duplicates)
USE_TEAM_CONSTRAINT = True        # 2-means on identity prototypes; never merge across teams
TEAM_FIT_MIN_FRAMES = 250
SPLIT_FINAL_CLASS_SWITCH = False  # Cell 12b: report only (True = old behaviour, adds IDs)

print("Checking paths...")
for p in [DEEP_EIOU_TRACKS_PATH, DEEP_EIOU_EMBS_PATH, BALL_DETECTION_CACHE_PATH, HOMOGRAPHY_PATH]:
    print(p, "->", "OK" if os.path.exists(p) else "MISSING")

# =============================================================================================
# v4: ZERO player ID switches (priority over the number of IDs; IDs are unioned manually later)
# =============================================================================================
STRICT_NO_SWITCH = True
USE_SHOT_CUTS = True
SHOT_CACHE_PATH = "/kaggle/working/shot_boundaries.npz"   # computed once from the video (Cell 7b), then reused
SHOT_HIST_THRESH = 0.35           # Bhattacharyya distance between consecutive frames' HSV histograms
SHOT_ADAPTIVE_K = 3.0             # ... and > K x the local median difference (ignores fast pans)
LINK_ACROSS_SHOTS = "pitch"       # False = never; "pitch" = only if the pitch position is continuous (below)
CONTACT_EXPAND = 0.2              # enlarge boxes by 20% (Deep-EIoU matches on expanded boxes too)
CONTACT_MERGE_GAP = 5             # contact frames closer than this form one contact run
DROP_CONTACT_SEGMENTS = True      # drop the ambiguous piece during a contact (who is who is uncertain there)
LINK_MIN_MARGIN = 0.1             # a link must beat the runner-up (both sides) by this much
REQUIRE_PITCH_FOR_LINK = True     # no link without valid pitch positions at both ends (no pixel-only links)

if STRICT_NO_SWITCH:
    SPLIT_MODE = "strict"
    CONTACT_IOU_THRESH = 0.05     # any real overlap of the enlarged boxes
    DRIFT_SPLIT_THRESH = 0.2      # was 0.3: cut at smaller appearance changes too
    MIN_LINK_SCORE = 0.6
    MAX_LINK_GAP_BY_CLASS = {**MAX_LINK_GAP_BY_CLASS, "player": 50}   # 2 s: only bridge short dropouts
    PLAYER_MERGE = False          # no identity-level merge -> you union IDs manually
    MIN_TRACK_LENGTH = 100        # was 300: pieces are shorter now; 300 would drop a large share of the data
    print("STRICT_NO_SWITCH: split=strict, drift=0.2, link score>=0.6 & margin>=0.1, player gap<=50, "
          "no merge, min ID length 100 frames")

# ---- v5: fewer pieces, still zero switches ----
TEAM_AWARE_CONTACTS = True        # cut only same-team player contacts (opponent swaps -> team_switch cut)
TEAM_SWITCH_CUT = True
TEAM_MIN_MARGIN = 0.05            # cosine margin between the two team centroids to trust a team label
TEAM_LOOKUP_FRAMES = 12           # a box takes the team label of the nearest embedding within this many frames
TEAM_SWITCH_SMOOTH = 5            # in embedding samples (every 5th frame -> 25 frames)
TEAM_SWITCH_MIN_RUN = 10          # a team run must last >= 10 samples (~2 s) to count
CROSS_SHOT_MAX_GAP = 10           # frames
CROSS_SHOT_MAX_DIST_M = 3.0
LINK_APPEARANCE_FLOOR_Q = 0.995   # a link's appearance must beat 99.5% of DIFFERENT same-team players (calibrated
                                  # in Cell 9 on co-visible pairs = guaranteed different people). None = off


## Cell 2 -- Load ball detections + homography cache
The ft3 detection cache is only used for the **ball** field of the output cache (ball model unchanged).

In [ ]:
with open(BALL_DETECTION_CACHE_PATH, "rb") as f:
    detection_cache = pickle.load(f)

with open(HOMOGRAPHY_PATH, "rb") as f:
    homography_cache = pickle.load(f)

print(f"Ball/detection cache loaded: {len(detection_cache)} frames")
n_valid_h = sum(1 for h in homography_cache if h is not None)
print(f"Homography cache loaded: {len(homography_cache)} frames, valid: {n_valid_h}/{len(homography_cache)}")

## Cell 5 -- Core functions: tracklets, splitting, pitch-space features, sparse global linking
Same functions as the BoT-SORT version, plus the selective-splitting helpers. `compute_tracklet_features`
now takes head/tail appearance from the stored (every-5th-frame) embeddings.

In [ ]:
def iou(box1, box2):
    xa1, ya1, xa2, ya2 = box1; xb1, yb1, xb2, yb2 = box2
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = (xa2 - xa1) * (ya2 - ya1) + (xb2 - xb1) * (yb2 - yb1) - inter
    return inter / union if union > 0 else 0.0


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])


def foot_anchor(bbox):
    x1, y1, x2, y2 = bbox
    return (x1 + x2) / 2, y2


def project_to_pitch(H, x, y):
    """Project an image-space point to pitch-space meters via homography."""
    pt = cv2.perspectiveTransform(np.array([[[x, y]]], dtype=np.float32), np.array(H)).reshape(2)
    return np.array([pt[0] / PX_PER_METER, pt[1] / PX_PER_METER])


def pitch_position(frame_idx, bbox, hcache):
    """Returns pitch-space (meters) position, or None if homography is unavailable for this frame."""
    if frame_idx == 0 or frame_idx >= len(hcache) or hcache[frame_idx] is None:
        return None
    fx, fy = foot_anchor(bbox)
    return project_to_pitch(hcache[frame_idx], fx, fy)


def _norm(v):
    n = float(np.linalg.norm(v))
    return v / n if n > 1e-12 else v


def _pitch_anchor(tl, frames_iter, hcache):
    """First frame (in iteration order) with a valid pitch position -> (frame, pos_m), else None."""
    for f in frames_iter:
        p = pitch_position(f, tl["bbox_by_frame"][f], hcache)
        if p is not None and np.all(np.isfinite(p)):
            return (f, p)
    return None


def reachable(anchor_a, anchor_b, max_speed_mps=None, slack_m=None):
    """True if a person seen at anchor_a can physically be at anchor_b. Unknown position -> True."""
    if anchor_a is None or anchor_b is None:
        return True
    v = MAX_SPEED_MPS if max_speed_mps is None else max_speed_mps
    s = POS_SLACK_M if slack_m is None else slack_m
    dt = max(abs(anchor_b[0] - anchor_a[0]), 1) / FPS
    return float(np.linalg.norm(anchor_b[1] - anchor_a[1])) <= v * dt + s


def team_label(emb):
    """v5: 0/1 = nearest team centroid, -1 = unsure (margin < TEAM_MIN_MARGIN) or no team model."""
    C = globals().get("TEAM_CENTROIDS")
    if C is None or emb is None:
        return -1
    s = C @ np.asarray(emb, dtype=np.float32)
    return int(np.argmax(s)) if abs(float(s[0] - s[1])) >= globals().get("TEAM_MIN_MARGIN", 0.05) else -1


def make_tracklet(entries, raw_track_id):
    """entries: list of (frame_idx, bbox, conf, cls, embedding), sorted by frame_idx."""
    frames = [e[0] for e in entries]
    bbox_by_frame = {e[0]: e[1] for e in entries}
    conf_by_frame = {e[0]: e[2] for e in entries}
    class_by_frame = {e[0]: e[3] for e in entries}
    embedding_by_frame = {e[0]: e[4] for e in entries}
    classes = [e[3] for e in entries]
    majority_class = max(set(classes), key=classes.count)
    return {
        "frames": frames, "bbox_by_frame": bbox_by_frame, "conf_by_frame": conf_by_frame,
        "class_by_frame": class_by_frame, "embedding_by_frame": embedding_by_frame,
        "class": majority_class, "raw_track_id": raw_track_id,
    }


def build_initial_tracklets(raw_tracks_by_frame):
    """Groups raw per-frame tracker output by raw_track_id. A raw ID's lifespan is already
    one continuous stretch with no ambiguity according to the tracker -- exactly what a
    tracklet should be before contact-based splitting. The frame-gap segmentation below is a
    no-op safety net for genuinely continuous raw tracker output (BotSort assigns a NEW raw ID
    after any real break, so an unbridged raw track should never contain an internal gap) --
    it is NOT the same bug as splitting a BRIDGED track on gaps, which would wrongly undo a
    deliberate cross-occlusion merge. This notebook loads the UNBRIDGED file, so that concern
    does not apply here."""
    raw_entries_by_id = defaultdict(list)
    for frame_idx, dets in raw_tracks_by_frame.items():
        for d in dets:
            raw_entries_by_id[d["raw_track_id"]].append(
                (frame_idx, d["bbox"], d["conf"], d["class"], d["embedding"])
            )

    tracklets = []
    for raw_id, entries in raw_entries_by_id.items():
        entries.sort(key=lambda e: e[0])
        segments, current = [], [entries[0]]
        for e in entries[1:]:
            if e[0] == current[-1][0] + 1:
                current.append(e)
            else:
                segments.append(current)
                current = [e]
        segments.append(current)
        for seg in segments:
            tracklets.append(make_tracklet(seg, raw_id))
    return tracklets


def find_contact_split_points(tracklets, iou_thresh, merge_gap=1):
    """Returns {tracklet_index: set of cut-boundary frames}. A cut boundary at frame f means
    end a chunk after frame f -- so the tracklet is split right before and right after each
    contact event, isolating the ambiguous contact frames into their own short segment instead
    of leaving them attached to a clean pre/post segment.

    Contact frames are grouped into contiguous runs (allowing gaps of up to merge_gap frames to
    still count as one run) before computing boundaries -- otherwise an extended contact (many
    consecutive overlapping frames) would get a cut point at EVERY overlapping frame, shredding
    the tracklet into single-frame fragments that then get dropped entirely by MIN_TRACKLET_LEN,
    silently deleting real player data for that whole stretch.
    """
    overlap_frames_by_pair = defaultdict(set)
    frame_to_tracklets = defaultdict(list)
    for idx, tl in enumerate(tracklets):
        for f in tl["frames"]:
            frame_to_tracklets[f].append(idx)

    for f, idxs in frame_to_tracklets.items():
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                a, b = tracklets[idxs[i]], tracklets[idxs[j]]
                if a["class"] != b["class"]:
                    continue
                if iou(a["bbox_by_frame"][f], b["bbox_by_frame"][f]) >= iou_thresh:
                    pair = tuple(sorted((idxs[i], idxs[j])))
                    overlap_frames_by_pair[pair].add(f)

    split_points = defaultdict(set)
    for (i, j), frames in overlap_frames_by_pair.items():
        frames = sorted(frames)
        runs, current = [], [frames[0]]
        for f in frames[1:]:
            if f - current[-1] <= merge_gap:
                current.append(f)
            else:
                runs.append(current)
                current = [f]
        runs.append(current)
        for run in runs:
            split_points[i].add(run[0] - 1)
            split_points[i].add(run[-1])
            split_points[j].add(run[0] - 1)
            split_points[j].add(run[-1])
    return split_points


def slice_tracklet(tl, frames):
    return {
        "frames": frames,
        "bbox_by_frame": {f: tl["bbox_by_frame"][f] for f in frames},
        "conf_by_frame": {f: tl["conf_by_frame"][f] for f in frames},
        "class_by_frame": {f: tl["class_by_frame"][f] for f in frames},
        "embedding_by_frame": {f: tl["embedding_by_frame"][f] for f in frames},
        "class": tl["class"],
        "raw_track_id": tl["raw_track_id"],
    }


def apply_splits(tracklets, split_points):
    new_tracklets = []
    for idx, tl in enumerate(tracklets):
        cuts = set(split_points.get(idx, []))
        if not cuts:
            new_tracklets.append(tl)
            continue
        chunks, current = [], []
        for f in tl["frames"]:
            current.append(f)
            if f in cuts:
                chunks.append(current)
                current = []
        if current:
            chunks.append(current)
        for chunk in chunks:
            if chunk:
                new_tracklets.append(slice_tracklet(tl, chunk))
    return new_tracklets


def fit_velocity(frames, bbox_by_frame):
    """Pixel-space fallback velocity fit -- only used when pitch-space projection is
    unavailable for enough of the window."""
    if len(frames) < 2:
        return None
    farr = np.array(frames, dtype=np.float64)
    centers = np.array([bbox_center(bbox_by_frame[f]) for f in frames])
    A = np.vstack([farr, np.ones_like(farr)]).T
    vx, cx = np.linalg.lstsq(A, centers[:, 0], rcond=None)[0]
    vy, cy = np.linalg.lstsq(A, centers[:, 1], rcond=None)[0]
    return np.array([vx, vy])


def fit_velocity_pitch(frames, bbox_by_frame, hcache):
    """Pitch-space (meters) velocity fit via homography. Returns None if fewer than 2 frames
    in the window have valid homography projection."""
    pts = []
    for f in frames:
        p = pitch_position(f, bbox_by_frame[f], hcache)
        if p is not None:
            pts.append((f, p))
    if len(pts) < 2:
        return None
    farr = np.array([p[0] for p in pts], dtype=np.float64)
    parr = np.array([p[1] for p in pts])
    A = np.vstack([farr, np.ones_like(farr)]).T
    vx, cx = np.linalg.lstsq(A, parr[:, 0], rcond=None)[0]
    vy, cy = np.linalg.lstsq(A, parr[:, 1], rcond=None)[0]
    return np.array([vx, vy])


def compute_tracklet_features(tl, window, hcache, emb_samples=None):
    """Head/tail features. Embeddings are stored every EMB_STRIDE frames (Deep-EIoU .npz), so the
    head/tail appearance uses the first/last `emb_samples` frames that HAVE an embedding, not the
    first/last `window` frames (which would hold only 1-2 samples at stride 5). Motion still uses
    the first/last `window` frames."""
    emb_samples = emb_samples or EMBED_SAMPLES
    frames = tl["frames"]
    head_frames = frames[:window]
    tail_frames = frames[-window:]

    emb_frames = [f for f in frames if tl["embedding_by_frame"].get(f) is not None]
    head_e, tail_e = emb_frames[:emb_samples], emb_frames[-emb_samples:]
    # v2: re-normalise the means (mean of unit vectors has norm < 1 -> dot product under-estimated cosine)
    tl["head_emb"] = (_norm(np.mean(np.stack([tl["embedding_by_frame"][f] for f in head_e]).astype(np.float32), axis=0))
                      if head_e else None)
    tl["tail_emb"] = (_norm(np.mean(np.stack([tl["embedding_by_frame"][f] for f in tail_e]).astype(np.float32), axis=0))
                      if tail_e else None)
    # v2: whole-tracklet appearance (sum of unit embeddings; used for slots, teams and identity merge)
    if emb_frames:
        tl["emb_sum"] = np.sum(np.stack([tl["embedding_by_frame"][f] for f in emb_frames]).astype(np.float32), axis=0)
        tl["n_emb"] = len(emb_frames)
        tl["proto"] = _norm(tl["emb_sum"])
    else:
        tl["emb_sum"], tl["n_emb"], tl["proto"] = None, 0, None
    tl["team"] = team_label(tl["proto"]) if tl["class"] == "player" else -1   # v5

    # pixel-space (fallback)
    tl["head_vel"] = fit_velocity(head_frames, tl["bbox_by_frame"])
    tl["tail_vel"] = fit_velocity(tail_frames, tl["bbox_by_frame"])
    tl["head_pos"] = bbox_center(tl["bbox_by_frame"][frames[0]])
    tl["tail_pos"] = bbox_center(tl["bbox_by_frame"][frames[-1]])

    # pitch-space (preferred) -- None if homography unavailable for this tracklet's boundary
    tl["head_vel_m"] = fit_velocity_pitch(head_frames, tl["bbox_by_frame"], hcache)
    tl["tail_vel_m"] = fit_velocity_pitch(tail_frames, tl["bbox_by_frame"], hcache)
    tl["head_pos_m"] = pitch_position(frames[0], tl["bbox_by_frame"][frames[0]], hcache)
    tl["tail_pos_m"] = pitch_position(frames[-1], tl["bbox_by_frame"][frames[-1]], hcache)
    # v2: nearest boundary frames WITH a valid homography (for the feasibility gate)
    tl["head_anchor"] = _pitch_anchor(tl, frames[:3 * window], hcache)
    tl["tail_anchor"] = _pitch_anchor(tl, frames[::-1][:3 * window], hcache)


def link_score(ti, tj, motion_weight, motion_norm_m, motion_norm_px, max_gap,
               velocity_extrapolation_cap_frames=250):
    gap = tj["frames"][0] - ti["frames"][-1]
    if gap <= 0 or gap > max_gap:
        return None
    # v2: hard physical-feasibility gate (last seen -> first seen must be reachable)
    if not reachable(ti.get("tail_anchor"), tj.get("head_anchor")):
        return None
    # v4: no link across a camera cut, and no link without pitch positions on both sides
    # v5: appearance must beat what two DIFFERENT teammates reach (calibrated floor, Cell 9)
    _amin = globals().get("LINK_MIN_APPEARANCE")
    if _amin is not None and (ti["tail_emb"] is None or tj["head_emb"] is None
                              or float(np.dot(ti["tail_emb"], tj["head_emb"])) < _amin):
        return None
    # v5: never across teams
    if ti.get("team", -1) >= 0 and tj.get("team", -1) >= 0 and ti["team"] != tj["team"]:
        return None
    shot = globals().get("SHOT_ID")
    if shot is not None and shot[ti["frames"][-1]] != shot[tj["frames"][0]]:
        mode = globals().get("LINK_ACROSS_SHOTS", True)
        if mode is False:
            return None
        if mode == "pitch":                       # across a camera cut only if the position is continuous
            a, b = ti.get("tail_anchor"), tj.get("head_anchor")
            if (gap > CROSS_SHOT_MAX_GAP or a is None or b is None
                    or float(np.linalg.norm(b[1] - a[1])) > CROSS_SHOT_MAX_DIST_M):
                return None
    if globals().get("REQUIRE_PITCH_FOR_LINK", False) and (ti.get("tail_anchor") is None or tj.get("head_anchor") is None):
        return None

    if ti["tail_emb"] is None or tj["head_emb"] is None:
        appearance_sim = 0.0
    else:
        appearance_sim = float(np.dot(ti["tail_emb"], tj["head_emb"]))

    # Cap how far a velocity estimate is allowed to extrapolate. A velocity fit over an
    # 8-frame window is noisy; multiplying that noise by a gap of tens of thousands of frames
    # can throw the prediction wildly off. Beyond this cap, extrapolation freezes at the
    # last-reached predicted position instead of continuing indefinitely.
    extrap_gap = min(gap, velocity_extrapolation_cap_frames)

    # Prefer pitch-space (metric, perspective-corrected) motion scoring. Pixel-space is a
    # fallback only for boundary frames where homography projection failed -- pixel distance
    # is not comparable in meaning across different gap sizes the way pitch meters are, so
    # this fallback is strictly weaker evidence and only used when pitch space is unavailable.
    if ti["tail_pos_m"] is not None and tj["head_pos_m"] is not None:
        vel = ti["tail_vel_m"] if ti["tail_vel_m"] is not None else np.zeros(2)
        predicted = ti["tail_pos_m"] + vel * extrap_gap
        dist = float(np.linalg.norm(predicted - tj["head_pos_m"]))
        motion_penalty = min(dist / motion_norm_m, 1.0)
    else:
        vel = ti["tail_vel"] if ti["tail_vel"] is not None else np.zeros(2)
        predicted = ti["tail_pos"] + vel * extrap_gap
        dist = float(np.linalg.norm(predicted - tj["head_pos"]))
        motion_penalty = min(dist / motion_norm_px, 1.0)

    return appearance_sim - motion_weight * motion_penalty


def link_tracklets_globally_sparse(tracklets, max_gap_by_class, min_link_score, motion_weight,
                                    motion_norm_m, motion_norm_px,
                                    velocity_extrapolation_cap_frames=250,
                                    default_max_gap=500, classes_to_skip=None, debug=False, min_margin=0.0):
    """
    Global (whole-match, not sliding-window) candidate consideration, resolved via greedy
    matching over a SPARSE candidate graph -- replaces dense per-class Hungarian assignment
    (scipy.optimize.linear_sum_assignment), which is ~O(n^3) and only tractable at
    3,000-frame-clip scale. At full-match scale (thousands of tracklets per class), dense
    Hungarian would never finish.

    max_gap_by_class: dict mapping class name -> max linkable gap in frames. Different classes
    fragment on very different timescales (goalkeeper/referee: long tracking dropouts, tens of
    thousands of frames; player: short contact-driven splits). A class missing from the dict
    falls back to default_max_gap.

    classes_to_skip: classes to exclude from appearance-based linking entirely (e.g.
    "goalkeeper", where identity resolution is instead handled via team-based merge -- see
    Cell 13 -- because appearance alone does not cleanly separate genuine same-person pairs
    for that class on real full-match data).

    Candidate generation uses bisect range queries against each class's max gap (not an
    all-pairs scan), so it stays near-linear in the number of tracklets plus the number of
    actually temporally-feasible pairs. Resolution is greedy (highest score first, each
    tracklet claimed at most once as a source and at most once as a target) rather than
    provably globally-optimal -- a deliberate tractability tradeoff, not an oversight.
    """
    classes_to_skip = classes_to_skip or set()
    links = {}
    by_class = defaultdict(list)
    for idx, tl in enumerate(tracklets):
        by_class[tl["class"]].append(idx)

    for cls, idxs in by_class.items():
        if cls in classes_to_skip:
            print(f"class={cls}: skipped (appearance-based linking disabled for this class)")
            continue

        max_gap = max_gap_by_class.get(cls, default_max_gap)
        idxs_by_start = sorted(idxs, key=lambda i: tracklets[i]["frames"][0])
        start_frames = [tracklets[i]["frames"][0] for i in idxs_by_start]

        candidates = []
        for i in idxs:
            end_frame = tracklets[i]["frames"][-1]
            lo = bisect.bisect_right(start_frames, end_frame)
            hi = bisect.bisect_right(start_frames, end_frame + max_gap)
            for k in range(lo, hi):
                j = idxs_by_start[k]
                if j == i:
                    continue
                score = link_score(tracklets[i], tracklets[j], motion_weight, motion_norm_m,
                                    motion_norm_px, max_gap, velocity_extrapolation_cap_frames)
                if score is not None:
                    candidates.append((score, i, j))

        if min_margin > 0:
            # v4: keep a link only if it is the MUTUAL best choice and beats the runner-up on both sides by
            # min_margin (all feasible pairs count as runner-ups, even those below min_link_score)
            best_out, best_in = defaultdict(list), defaultdict(list)
            for s, i, j in candidates:
                best_out[i].append((s, j)); best_in[j].append((s, i))
            def _top2(lst):
                lst.sort(key=lambda t: -t[0])
                return lst[0], (lst[1][0] if len(lst) > 1 else -np.inf)
            top_out = {i: _top2(v) for i, v in best_out.items()}
            top_in = {j: _top2(v) for j, v in best_in.items()}
            kept, n_ambiguous = [], 0
            for s, i, j in candidates:
                if s < min_link_score:
                    continue
                (so, jo), ro = top_out[i]
                (si, io), ri = top_in[j]
                if jo == j and io == i and s - ro >= min_margin and s - ri >= min_margin:
                    kept.append((s, i, j))
                else:
                    n_ambiguous += 1
            print(f"class={cls}: {n_ambiguous} candidate links above threshold rejected as ambiguous (margin < {min_margin})")
            candidates = kept
        else:
            candidates = [t for t in candidates if t[0] >= min_link_score]

        candidates.sort(key=lambda c: -c[0])
        used_as_source, used_as_target = set(), set()
        n_class_links = 0
        for score, i, j in candidates:
            if i in used_as_source or j in used_as_target:
                continue
            links[i] = j
            used_as_source.add(i)
            used_as_target.add(j)
            n_class_links += 1
            if debug:
                print(f"  [link] class={cls}: tracklet {i} (ends frame {tracklets[i]['frames'][-1]}) "
                      f"-> tracklet {j} (starts frame {tracklets[j]['frames'][0]}), score={score:.3f}")
        print(f"class={cls}: {len(idxs)} tracklets, {len(candidates)} candidate pairs (max_gap={max_gap}), {n_class_links} links accepted")

    return links


# ---------------------------------------------------------------------------------------------
# Selective splitting (replaces "split at every contact" and the manual track-2/41 splits).
# Each function returns the frames f where a cut must happen BEFORE f.
# ---------------------------------------------------------------------------------------------

def class_switch_cut_frames(frames, classes, smooth_frames, min_run_frames):
    """Sustained class switches inside one track (e.g. referee -> player), found automatically.
    Per-frame classes are majority-smoothed over `smooth_frames`; runs of the smoothed label
    shorter than `min_run_frames` are treated as detector flicker and absorbed into the
    neighbouring run. Every remaining run boundary is a cut."""
    labels = np.asarray(classes)
    uniq = sorted(set(labels.tolist()))
    if len(uniq) < 2:
        return []
    onehot = np.stack([(labels == u).astype(np.float32) for u in uniq], axis=1)
    counts = pd.DataFrame(onehot).rolling(smooth_frames, center=True, min_periods=1).sum().values
    smoothed = np.asarray(uniq, dtype=object)[counts.argmax(axis=1)]

    runs, start = [], 0
    for i in range(1, len(smoothed) + 1):
        if i == len(smoothed) or smoothed[i] != smoothed[start]:
            runs.append([start, i, smoothed[start]])
            start = i
    merged = []
    for r in runs:
        if merged and (r[1] - r[0] < min_run_frames or r[2] == merged[-1][2]):
            merged[-1][1] = r[1]            # flicker (or same label again): absorb
        else:
            merged.append(r)
    if len(merged) > 1 and merged[0][1] - merged[0][0] < min_run_frames:
        merged[1][0] = merged[0][0]         # short first run: absorb into the next one
        merged.pop(0)
    return [frames[r[0]] for r in merged[1:]]


def jump_cut_frames(frames, bbox_by_frame, max_gap, jump_ratio):
    """Abrupt box jumps between near-consecutive observations (ID swap or camera cut). Both are
    unsafe to keep inside one tracklet, so both are cut."""
    cuts = []
    for k in range(1, len(frames)):
        f0, f1 = frames[k - 1], frames[k]
        if f1 - f0 > max_gap:
            continue
        b0, b1 = bbox_by_frame[f0], bbox_by_frame[f1]
        h = ((b0[3] - b0[1]) + (b1[3] - b1[1])) / 2
        if np.linalg.norm(bbox_center(b1) - bbox_center(b0)) / max(h, 1.0) > jump_ratio:
            cuts.append(f1)
    return cuts


def drift_cut_frames(frames, embedding_by_frame, win_frames, drift_thresh, min_samples=3):
    """Sharp appearance change between adjacent time windows of the same track."""
    wins = defaultdict(list)
    for f in frames:
        e = embedding_by_frame.get(f)
        if e is not None:
            wins[f // win_frames].append((f, e))
    keys = sorted(k for k, v in wins.items() if len(v) >= min_samples)
    means = {}
    for k in keys:
        m = np.mean(np.stack([e for _, e in wins[k]]).astype(np.float32), axis=0)
        means[k] = m / max(np.linalg.norm(m), 1e-12)
    cuts = []
    for a, b in zip(keys[:-1], keys[1:]):
        if 1.0 - float(np.dot(means[a], means[b])) > drift_thresh:
            first_f_b = min(f for f, _ in wins[b])
            cuts.append(first_f_b)
    return cuts


def pitch_jump_cut_frames(tl, hcache, max_gap=JUMP_MAX_GAP, speed=None, slack=None, ctx=None):
    """Cut where the track's pitch position teleports: median position of the ctx observations
    before f vs the ctx observations from f on is farther than speed * dt + slack. Catches a
    Deep-EIoU track hopping to another player (typically across a camera cut), which image-space
    jumps and appearance drift miss when both players wear the same kit."""
    speed = PITCH_JUMP_SPEED_MPS if speed is None else speed
    slack = PITCH_JUMP_SLACK_M if slack is None else slack
    ctx = PITCH_JUMP_CTX if ctx is None else ctx
    F, P = [], []
    for f in tl["frames"]:
        p = pitch_position(f, tl["bbox_by_frame"][f], hcache)
        if p is not None and np.all(np.isfinite(p)):
            F.append(f); P.append(p)
    if len(F) < 2 * ctx:
        return []
    F, P = np.asarray(F), np.asarray(P)
    cuts, last_cut = [], -10**9
    for k in range(ctx, len(F) - ctx + 1):
        if F[k] - F[k - 1] > max_gap or k - last_cut < 2 * ctx:
            continue
        a = np.median(P[k - ctx:k], axis=0)
        b = np.median(P[k:k + ctx], axis=0)
        dt = (F[k + ctx - 1] - F[k - ctx]) / FPS
        if np.linalg.norm(b - a) > speed * dt + slack:
            # the medians confirm a real teleport; place the cut at the largest single step nearby
            js = np.arange(k - ctx + 1, k + ctx)
            j = int(js[np.argmax(np.linalg.norm(P[js] - P[js - 1], axis=1))])
            if j - last_cut >= ctx:
                cuts.append(int(F[j])); last_cut = j
    return cuts


def shot_cut_frames(frames):
    """v4: frames where a tracklet enters a new camera shot (cut BEFORE these frames)."""
    shot = globals().get("SHOT_ID")
    if shot is None or len(frames) < 2:
        return []
    s = shot[np.asarray(frames)]
    return [frames[k] for k in np.where(s[1:] != s[:-1])[0] + 1]


def contact_runs_by_tracklet(tracklets, iou_thresh, expand, target_classes=("player",), merge_gap=5,
                             partner_classes=None, team_at=None):
    """v4: {tracklet idx: [(start, end), ...]} contact runs. A frame is a contact for a target-class tracklet when
    its box, enlarged by `expand`, overlaps (IoU >= iou_thresh) the enlarged box of ANY other tracklet in that frame."""
    by_frame = defaultdict(list)
    for idx, tl in enumerate(tracklets):
        for f in tl["frames"]:
            by_frame[f].append(idx)
    contact_frames = defaultdict(list)
    for f, idxs in by_frame.items():
        if len(idxs) < 2:
            continue
        B = np.array([tracklets[i]["bbox_by_frame"][f] for i in idxs], dtype=np.float64)
        w, h = (B[:, 2] - B[:, 0]) * expand / 2, (B[:, 3] - B[:, 1]) * expand / 2
        E = np.stack([B[:, 0] - w, B[:, 1] - h, B[:, 2] + w, B[:, 3] + h], axis=1)
        iw = np.clip(np.minimum(E[:, None, 2], E[None, :, 2]) - np.maximum(E[:, None, 0], E[None, :, 0]), 0, None)
        ih = np.clip(np.minimum(E[:, None, 3], E[None, :, 3]) - np.maximum(E[:, None, 1], E[None, :, 1]), 0, None)
        inter = iw * ih
        area = (E[:, 2] - E[:, 0]) * (E[:, 3] - E[:, 1])
        io = inter / np.maximum(area[:, None] + area[None, :] - inter, 1e-9)
        np.fill_diagonal(io, 0.0)
        if partner_classes is not None:            # v5: only these classes count as a contact partner
            ok = np.array([tracklets[i]["class"] in partner_classes for i in idxs])
            io = io * ok[None, :]
        if team_at is not None:                    # v5: only same-team (or unknown-team) contacts count
            T = np.array([team_at(i, f) for i in idxs])
            same = (T[:, None] == T[None, :]) | (T[:, None] < 0) | (T[None, :] < 0)
            io = io * same
        for k in np.where((io >= iou_thresh).any(axis=1))[0]:
            if tracklets[idxs[k]]["class"] in target_classes:
                contact_frames[idxs[k]].append(f)
    runs = {}
    for idx, fr in contact_frames.items():
        fr.sort()
        rr, cur = [], [fr[0], fr[0]]
        for f in fr[1:]:
            if f - cur[1] <= merge_gap:
                cur[1] = f
            else:
                rr.append(tuple(cur)); cur = [f, f]
        rr.append(tuple(cur))
        runs[idx] = rr
    return runs


def split_tracklet_before(tl, cut_frames):
    """Split a tracklet so that each frame in cut_frames starts a new chunk."""
    cuts = set(cut_frames)
    chunks, current = [], []
    for f in tl["frames"]:
        if f in cuts and current:
            chunks.append(current)
            current = []
        current.append(f)
    if current:
        chunks.append(current)
    out = []
    for ch in chunks:
        piece = slice_tracklet(tl, ch)
        cls_list = [piece["class_by_frame"][f] for f in ch]
        piece["class"] = max(set(cls_list), key=cls_list.count)   # re-vote class per chunk
        out.append(piece)
    return out


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[rx] = ry


print("Core functions ready (incl. selective splitting).")

## Cell 6 -- Unit tests: global linking + selective splitting
The original linking test (3 fragments chained, decoy excluded), plus a test that the class-switch
detector cuts a sustained referee -> player switch and ignores short flicker.

In [ ]:
def _run_global_linking_unit_test():
    print("Running global linking unit test...")

    rng = np.random.default_rng(7)
    emb_player = rng.normal(size=512); emb_player /= np.linalg.norm(emb_player)
    emb_decoy  = rng.normal(size=512); emb_decoy  /= np.linalg.norm(emb_decoy)

    identity_h = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    test_hcache = [identity_h] * 200

    def make(frames, x_start, vx, emb, cls="player"):
        bbox_by_frame, conf_by_frame, class_by_frame, embedding_by_frame = {}, {}, {}, {}
        for k, f in enumerate(frames):
            x = x_start + vx * k
            bbox_by_frame[f] = [x, 100, x + 40, 180]
            conf_by_frame[f] = 0.9
            class_by_frame[f] = cls
            embedding_by_frame[f] = emb
        return {
            "frames": frames, "bbox_by_frame": bbox_by_frame, "conf_by_frame": conf_by_frame,
            "class_by_frame": class_by_frame, "embedding_by_frame": embedding_by_frame, "class": cls,
            "raw_track_id": "synthetic",
        }

    frag1 = make(list(range(0, 20)), x_start=100, vx=2, emb=emb_player)
    frag2 = make(list(range(40, 60)), x_start=100 + 2*19 + 2*20, vx=2, emb=emb_player)
    frag3 = make(list(range(90, 110)), x_start=100 + 2*19 + 2*20 + 2*19 + 2*30, vx=2, emb=emb_player)
    decoy = make(list(range(45, 65)), x_start=500, vx=-1, emb=emb_decoy)

    test_tracklets = [frag1, frag2, frag3, decoy]
    for tl in test_tracklets:
        compute_tracklet_features(tl, window=8, hcache=test_hcache)

    # motion_norm_m=30.0 matches the original test's motion_norm_px=300 scaled by PX_PER_METER=10
    links = link_tracklets_globally_sparse(test_tracklets, max_gap_by_class={"player": 50},
                                            min_link_score=0.3, motion_weight=0.3,
                                            motion_norm_m=30.0, motion_norm_px=300.0,
                                            velocity_extrapolation_cap_frames=50, debug=False)

    test_uf = UnionFind(len(test_tracklets))
    for i, j in links.items():
        test_uf.union(i, j)
    groups = defaultdict(list)
    for idx in range(len(test_tracklets)):
        groups[test_uf.find(idx)].append(idx)

    print(f"  links: {links}")
    print(f"  groups: {dict(groups)}")

    group_sizes = sorted(len(v) for v in groups.values())
    assert group_sizes == [1, 3], f"FAIL: expected groups of size [1,3], got {group_sizes}"
    real_group = [v for v in groups.values() if len(v) == 3][0]
    assert set(real_group) == {0, 1, 2}, f"FAIL: wrong tracklets grouped: {real_group}"

    print("PASS: 3 fragments of the real player correctly chained; decoy correctly excluded.")
    print("")


_run_global_linking_unit_test()



def _run_selective_split_unit_test():
    print("Running selective split unit test...")
    frames = list(range(400))
    # referee for 200 frames with 10 frames of player flicker, then a sustained switch to player
    classes = ["referee"] * 200 + ["player"] * 200
    for f in range(80, 90):
        classes[f] = "player"
    cuts = class_switch_cut_frames(frames, classes, CLASS_SWITCH_SMOOTH_FRAMES, CLASS_SWITCH_MIN_RUN)
    print(f"  class-switch cuts: {cuts}")
    assert len(cuts) == 1 and abs(cuts[0] - 200) <= CLASS_SWITCH_SMOOTH_FRAMES, f"FAIL: expected one cut near 200, got {cuts}"

    bboxes = {f: [100 + f, 100, 140 + f, 180] for f in frames}
    bboxes[300] = [900, 100, 940, 180]
    jcuts = jump_cut_frames(frames, bboxes, JUMP_MAX_GAP, JUMP_RATIO)
    assert jcuts == [300, 301], f"FAIL: expected jumps at [300, 301], got {jcuts}"

    rng = np.random.default_rng(0)
    ea = rng.normal(size=512); ea /= np.linalg.norm(ea)
    eb = rng.normal(size=512); eb /= np.linalg.norm(eb)
    embs = {f: (ea if f < 250 else eb) for f in frames if f % 5 == 0}
    dcuts = drift_cut_frames(frames, embs, DRIFT_WINDOW_FRAMES, DRIFT_SPLIT_THRESH)
    assert dcuts == [250], f"FAIL: expected drift cut at 250, got {dcuts}"
    print("PASS: sustained switch cut once, flicker ignored; jump and drift cuts at the right frames.")


_run_selective_split_unit_test()


def _run_margin_unit_test():
    print("Running ambiguity-margin unit test...")
    rng = np.random.default_rng(3)
    e = rng.normal(size=512); e /= np.linalg.norm(e)
    n2 = rng.normal(size=512); n2 /= np.linalg.norm(n2)
    e2 = e + 0.15 * n2; e2 /= np.linalg.norm(e2)                          # a look-alike teammate (cos ~0.99)
    H = np.eye(3, dtype=np.float32)
    hc = [H] * 200
    def mk(frames, x, emb):
        return {"frames": frames, "bbox_by_frame": {f: [x, 100, x + 30, 160] for f in frames},
                "conf_by_frame": {f: .9 for f in frames}, "class_by_frame": {f: "player" for f in frames},
                "embedding_by_frame": {f: emb for f in frames}, "class": "player", "raw_track_id": "t"}
    a = mk(list(range(1, 40)), 100, e)
    b1 = mk(list(range(45, 80)), 110, e)     # true continuation
    b2 = mk(list(range(46, 80)), 130, e2)    # look-alike 2 m away -> ambiguous
    for tl in (a, b1, b2):
        compute_tracklet_features(tl, window=8, hcache=hc)
    kw = dict(max_gap_by_class={"player": 50}, min_link_score=0.3, motion_weight=0.3, motion_norm_m=8.0,
              motion_norm_px=300.0, velocity_extrapolation_cap_frames=25)
    loose = link_tracklets_globally_sparse([a, b1, b2], **kw)
    strict = link_tracklets_globally_sparse([a, b1, b2], min_margin=0.1, **kw)
    assert loose == {0: 1}, f"FAIL loose: {loose}"
    assert strict == {}, f"FAIL strict: {strict}"
    print("PASS: without margin the look-alike case is linked; with margin it is left for manual union.")


_run_margin_unit_test()


## Cell 7 -- Load Deep-EIoU tracks and embeddings, build tracklets
One tracklet per Deep-EIoU track. Tracks are **not** split at frame gaps: Deep-EIoU only resumes a
track after a short loss (<= 2 s), and that re-association is kept.

In [ ]:
tracks_df = pd.read_parquet(DEEP_EIOU_TRACKS_PATH)
tracks_df = tracks_df[tracks_df.cls.isin(list(CLS_ID_TO_NAME))].sort_values(["track_id", "frame"])
n_frames = max(int(tracks_df.frame.max()) + 1, len(homography_cache))
print(f"Deep-EIoU rows: {len(tracks_df):,} | tracks: {tracks_df.track_id.nunique()} | frames: {n_frames}")

z = np.load(DEEP_EIOU_EMBS_PATH)
emb_lookup = {(int(f), int(t)): i for i, (f, t) in enumerate(zip(z["frame"], z["track_id"]))}
emb_array = z["emb"]                           # float16, L2-normalised per sample
print(f"Stored embeddings: {len(emb_lookup):,}")

initial_tracklets = []
for tid, g in tracks_df.groupby("track_id", sort=False):
    entries = []
    for f, x1, y1, x2, y2, conf, c in zip(g.frame.values, g.x1.values, g.y1.values, g.x2.values,
                                          g.y2.values, g.score.values, g.cls.values):
        k = emb_lookup.get((int(f), int(tid)))
        entries.append((int(f), [float(x1), float(y1), float(x2), float(y2)], float(conf),
                        CLS_ID_TO_NAME[int(c)], emb_array[k] if k is not None else None))
    initial_tracklets.append(make_tracklet(entries, int(tid)))

no_emb = sum(1 for tl in initial_tracklets if all(v is None for v in tl["embedding_by_frame"].values()))
print(f"Initial tracklets: {len(initial_tracklets)} | without any stored embedding: {no_emb}")
print("By class:", dict(Counter(tl["class"] for tl in initial_tracklets)))

## Cell 7b -- Camera-cut (shot boundary) detection
One sequential pass over the video (~5-10 min), cached to `SHOT_CACHE_PATH` so later runs skip it. A boundary is a
frame whose HSV-histogram distance to the previous frame is above `SHOT_HIST_THRESH` **and** `SHOT_ADAPTIVE_K` x the
local median (so fast pans do not count). Check the printed shot-length stats: broadcast play shots are usually
several seconds to minutes; many 1-2 frame "shots" mean the threshold is too low.

In [ ]:
from tqdm.auto import tqdm

SHOT_ID = None
if USE_SHOT_CUTS:
    if os.path.exists(SHOT_CACHE_PATH):
        shot_diff = np.load(SHOT_CACHE_PATH)["diff"]
        print(f"Loaded frame differences from {SHOT_CACHE_PATH}")
    else:
        cap = cv2.VideoCapture(VIDEO_PATH)
        assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
        shot_diff = np.zeros(n_frames + 1, dtype=np.float32)
        prev_hist = None
        for f in tqdm(range(n_frames), desc="Shot detection", unit="frame", mininterval=1.0):
            ok, img = cap.read()
            if not ok:
                print(f"video ended at frame {f}")
                break
            hsv = cv2.cvtColor(cv2.resize(img, (160, 90), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2HSV)
            hist = cv2.calcHist([hsv], [0, 1], None, [32, 16], [0, 180, 0, 256])
            cv2.normalize(hist, hist)
            if prev_hist is not None:
                shot_diff[f] = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_BHATTACHARYYA)
            prev_hist = hist
        cap.release()
        np.savez(SHOT_CACHE_PATH, diff=shot_diff)
        print(f"Saved frame differences to {SHOT_CACHE_PATH}")

    local_med = pd.Series(shot_diff).rolling(25, center=True, min_periods=1).median().values
    is_cut = (shot_diff > SHOT_HIST_THRESH) & (shot_diff > SHOT_ADAPTIVE_K * np.maximum(local_med, 1e-3))
    is_cut[0] = False
    SHOT_ID = np.cumsum(is_cut).astype(np.int64)
    if len(SHOT_ID) < n_frames + 1:
        SHOT_ID = np.pad(SHOT_ID, (0, n_frames + 1 - len(SHOT_ID)), mode="edge")
    cut_frames_all = np.where(is_cut)[0]
    shot_lens = np.diff(np.concatenate([[0], cut_frames_all, [n_frames]]))
    print(f"Frame-difference quantiles (50/90/99/99.9%): {np.round(np.quantile(shot_diff, [.5, .9, .99, .999]), 3)}")
    print(f"Shot boundaries: {len(cut_frames_all)} | shots: {len(shot_lens)} | shot length (s) "
          f"median {np.median(shot_lens) / FPS:.1f}, 10% {np.quantile(shot_lens, .1) / FPS:.1f}, "
          f"max {shot_lens.max() / FPS:.1f} | shots < 1 s: {int((shot_lens < FPS).sum())}")
    print(f"First boundaries (frame): {cut_frames_all[:15].tolist()}")
else:
    print("USE_SHOT_CUTS=False: no shot cuts.")


## Cell 7c -- Team labels from embeddings (for team-aware contacts, team-switch cuts, no cross-team links)
2-means on a sample of player embeddings. Check: the two centroids should be far apart (low cosine) and the
max number of same-team player boxes in one frame should be about 10-11. If not, set `TEAM_AWARE_CONTACTS = False`
and `TEAM_SWITCH_CUT = False` (v4 behaviour).

In [ ]:
from sklearn.cluster import KMeans

TEAM_CENTROIDS = None
if TEAM_AWARE_CONTACTS or TEAM_SWITCH_CUT:
    _e = pd.DataFrame({"frame": z["frame"].astype(np.int64), "track_id": z["track_id"].astype(np.int64),
                       "row": np.arange(len(z["frame"]))})
    _e = _e.merge(tracks_df[["frame", "track_id", "cls"]], on=["frame", "track_id"], how="inner")
    _rows = _e.row.values[_e.cls.values == 1]                      # player class
    _rng = np.random.default_rng(0)
    _fit = _rng.choice(_rows, size=min(60000, len(_rows)), replace=False)
    _X = emb_array[_fit].astype(np.float32)
    _km = KMeans(2, n_init=10, random_state=0).fit(_X)
    TEAM_CENTROIDS = np.stack([_norm(c) for c in _km.cluster_centers_]).astype(np.float32)
    print(f"Team model: {len(_fit)} player embeddings, sizes {dict(Counter(_km.labels_.tolist()))}, "
          f"centroid cosine {float(TEAM_CENTROIDS[0] @ TEAM_CENTROIDS[1]):.3f}")

    # per-tracklet team label on every embedded frame, majority-smoothed over 5 samples
    for tl in initial_tracklets:
        F = sorted(f for f, e in tl["embedding_by_frame"].items() if e is not None)
        if not F:
            tl["_team_f"], tl["_team_l"] = [], np.zeros(0, dtype=np.int64); continue
        S = np.stack([tl["embedding_by_frame"][f] for f in F]).astype(np.float32) @ TEAM_CENTROIDS.T
        L = np.where(np.abs(S[:, 0] - S[:, 1]) >= TEAM_MIN_MARGIN, S.argmax(1), -1)
        c0 = pd.Series((L == 0).astype(float)).rolling(5, center=True, min_periods=1).sum().values
        c1 = pd.Series((L == 1).astype(float)).rolling(5, center=True, min_periods=1).sum().values
        tl["_team_f"] = F
        tl["_team_l"] = np.where(c0 > c1, 0, np.where(c1 > c0, 1, -1))

    def team_at(idx, f, _src=None):
        tl = (initial_tracklets if _src is None else _src)[idx]
        F = tl.get("_team_f")
        if not F:
            return -1
        k = bisect.bisect_left(F, f)
        best, bd = -1, TEAM_LOOKUP_FRAMES + 1
        for kk in (k - 1, k):
            if 0 <= kk < len(F) and abs(F[kk] - f) < bd:
                bd, best = abs(F[kk] - f), int(tl["_team_l"][kk])
        return best

    # sanity: max same-team player boxes in one frame (embedded frames only)
    _cnt = defaultdict(Counter)
    for tl in initial_tracklets:
        if tl["class"] != "player":
            continue
        for f, l in zip(tl["_team_f"], tl["_team_l"]):
            if l >= 0:
                _cnt[f][int(l)] += 1
    _mx = [max((c[t] for c in _cnt.values()), default=0) for t in (0, 1)]
    _p99 = [int(np.quantile([c[t] for c in _cnt.values()], .99)) if _cnt else 0 for t in (0, 1)]
    print(f"Max same-team player boxes in one frame: {_mx} (99th pct {_p99}) -- about 10-11 expected")
else:
    team_at = None
    print("Team model disabled.")


## Cell 8 -- Split where an ID switch could happen, then filter short tracklets
`strict` (v4 default): selective cuts (class switch, image jump, drift, pitch teleport) **plus every camera cut and every player contact**; contact pieces are dropped.
`selective`: without shots/contacts. `all_contacts`: old behaviour.

In [ ]:
cut_reasons = Counter()
tracks_cut = Counter()

n_contact_pieces_dropped = n_contact_frames_dropped = 0
if SPLIT_MODE in ("selective", "strict"):
    from tqdm.auto import tqdm
    contact_runs = {}
    if SPLIT_MODE == "strict":
        contact_runs = contact_runs_by_tracklet(
            initial_tracklets, CONTACT_IOU_THRESH, CONTACT_EXPAND, target_classes=("player",),
            merge_gap=CONTACT_MERGE_GAP,
            partner_classes=("player",) if TEAM_AWARE_CONTACTS else None,
            team_at=team_at if (TEAM_AWARE_CONTACTS and TEAM_CENTROIDS is not None) else None)
        n_pf = sum(len(tl["frames"]) for tl in initial_tracklets if tl["class"] == "player")
        n_cf = sum(sum(min(b, 10**9) - a + 1 for a, b in rr) for rr in contact_runs.values())
        print(f"Contact runs: {sum(len(v) for v in contact_runs.values())} on {len(contact_runs)} player tracks "
              f"(~{n_cf / max(n_pf, 1):.1%} of player frames)")
    split_tracklets = []
    for idx, tl in enumerate(tqdm(initial_tracklets, desc=f"Splitting ({SPLIT_MODE})")):
        frames = tl["frames"]
        c_cuts = class_switch_cut_frames(frames, [tl["class_by_frame"][f] for f in frames],
                                         CLASS_SWITCH_SMOOTH_FRAMES, CLASS_SWITCH_MIN_RUN)
        j_cuts = jump_cut_frames(frames, tl["bbox_by_frame"], JUMP_MAX_GAP, JUMP_RATIO)
        d_cuts = drift_cut_frames(frames, tl["embedding_by_frame"], DRIFT_WINDOW_FRAMES, DRIFT_SPLIT_THRESH)
        p_cuts = pitch_jump_cut_frames(tl, homography_cache)
        s_cuts, k_cuts, cframes, t_cuts = [], [], set(), []
        if TEAM_SWITCH_CUT and TEAM_CENTROIDS is not None and tl["class"] == "player" and len(tl.get("_team_f", [])) > 1:
            _L = pd.Series(tl["_team_l"]).replace(-1, np.nan).ffill().bfill()
            if _L.notna().all():
                t_cuts = class_switch_cut_frames(list(tl["_team_f"]), _L.astype(int).tolist(),
                                                 TEAM_SWITCH_SMOOTH, TEAM_SWITCH_MIN_RUN)
        if SPLIT_MODE == "strict":
            s_cuts = shot_cut_frames(frames)
            fr_arr = np.asarray(frames)
            for a, b in contact_runs.get(idx, []):
                k_cuts.append(a)                                   # contact starts -> new piece
                nxt = fr_arr[fr_arr > b]
                if len(nxt):
                    k_cuts.append(int(nxt[0]))                     # contact ends -> new piece
                cframes.update(fr_arr[(fr_arr >= a) & (fr_arr <= b)].tolist())
        for name, cuts in (("class_switch", c_cuts), ("jump", j_cuts), ("drift", d_cuts), ("pitch_jump", p_cuts),
                           ("shot", s_cuts), ("contact", k_cuts), ("team_switch", t_cuts)):
            cut_reasons[name] += len(cuts)
            tracks_cut[name] += int(bool(cuts))
        cuts = set(c_cuts) | set(j_cuts) | set(d_cuts) | set(p_cuts) | set(s_cuts) | set(k_cuts) | set(t_cuts)
        pieces = split_tracklet_before(tl, cuts) if cuts else [tl]
        if DROP_CONTACT_SEGMENTS and cframes:
            kept_pieces = []
            for pc in pieces:
                if sum(f in cframes for f in pc["frames"]) >= 0.5 * len(pc["frames"]):
                    n_contact_pieces_dropped += 1; n_contact_frames_dropped += len(pc["frames"])
                else:
                    kept_pieces.append(pc)
            pieces = kept_pieces
        split_tracklets.extend(pieces)
elif SPLIT_MODE == "all_contacts":
    split_points = find_contact_split_points(initial_tracklets, CONTACT_IOU_THRESH)
    split_tracklets = apply_splits(initial_tracklets, split_points)
    cut_reasons["contact"] = sum(len(v) for v in split_points.values())
else:
    split_tracklets = list(initial_tracklets)

lens_raw = sorted((len(tl["frames"]) for tl in initial_tracklets), reverse=True)
print(f"Longest raw Deep-EIoU tracks (frames): {lens_raw[:10]}  (> ~3000 frames = 2 min is suspicious on broadcast)")
print(f"Split mode: {SPLIT_MODE}")
print(f"Cuts by reason: {dict(cut_reasons)} | tracks affected: {dict(tracks_cut)}")
print(f"Tracklets after splitting: {len(split_tracklets)} (from {len(initial_tracklets)})")
if n_contact_pieces_dropped:
    print(f"Contact pieces dropped: {n_contact_pieces_dropped} ({n_contact_frames_dropped} frames)")
lens_split = sorted((len(tl["frames"]) for tl in split_tracklets), reverse=True)
print(f"Longest tracklets after splitting (frames): {lens_split[:10]}")

pre_filter_count = len(split_tracklets)
split_tracklets = [tl for tl in split_tracklets if len(tl["frames"]) >= MIN_TRACKLET_LEN]
print(f"Dropped {pre_filter_count - len(split_tracklets)} tracklets shorter than MIN_TRACKLET_LEN={MIN_TRACKLET_LEN}")
print(f"Tracklets going into global linking: {len(split_tracklets)}")
print("By class:", dict(Counter(tl["class"] for tl in split_tracklets)))

## Cell 9 -- Compute features, then sparse global linking (players only when officials are forced)
Goalkeeper and referee tracklets are **not** chained here when `FORCE_OFFICIALS` is on: they go to the fixed slots in Cell 9b.

In [ ]:
for tl in split_tracklets:
    compute_tracklet_features(tl, EMBED_WINDOW, homography_cache)

n_missing_pitch = sum(1 for tl in split_tracklets if tl["head_anchor"] is None or tl["tail_anchor"] is None)
n_missing_emb = sum(1 for tl in split_tracklets if tl["head_emb"] is None or tl["tail_emb"] is None)
print(f"Computed features for {len(split_tracklets)} tracklets.")
print(f"No valid pitch anchor at a boundary (feasibility gate skipped): {n_missing_pitch}")
print(f"Missing head/tail embedding (appearance score = 0, will rarely link): {n_missing_emb}")

# v5: calibrate the appearance floor on co-visible same-team player tracklets (certainly different people)
LINK_MIN_APPEARANCE = None
if STRICT_NO_SWITCH and LINK_APPEARANCE_FLOOR_Q:
    _pl = [i for i, tl in enumerate(split_tracklets) if tl["class"] == "player" and tl["head_emb"] is not None
           and tl["tail_emb"] is not None and tl.get("team", -1) >= 0]
    _bk = defaultdict(set)
    for i in _pl:
        for f in split_tracklets[i]["frames"][::25]:
            _bk[f // 25].add(i)
    _pairs = set()
    for ids_ in _bk.values():
        ids_ = sorted(ids_)
        for a_ in range(len(ids_)):
            for b_ in range(a_ + 1, len(ids_)):
                _pairs.add((ids_[a_], ids_[b_]))
    _sims = []
    for i, j in _pairs:
        ti_, tj_ = split_tracklets[i], split_tracklets[j]
        if ti_["team"] != tj_["team"]:
            continue
        if min(ti_["frames"][-1], tj_["frames"][-1]) - max(ti_["frames"][0], tj_["frames"][0]) < 10:
            continue                                            # must really be on screen together
        _sims += [float(np.dot(ti_["tail_emb"], tj_["head_emb"])), float(np.dot(tj_["tail_emb"], ti_["head_emb"]))]
    if len(_sims) >= 100:
        LINK_MIN_APPEARANCE = float(np.quantile(_sims, LINK_APPEARANCE_FLOOR_Q))
        print(f"Different-teammate appearance sims ({len(_sims)} from co-visible pairs): "
              f"median {np.median(_sims):.3f}, 99% {np.quantile(_sims, .99):.3f} -> link floor "
              f"LINK_MIN_APPEARANCE = {LINK_MIN_APPEARANCE:.3f}")
    else:
        print(f"Not enough co-visible same-team pairs ({len(_sims)}) to calibrate; no appearance floor.")

LINK_SKIP = {"goalkeeper", "referee"} if FORCE_OFFICIALS else set()
accepted_links = link_tracklets_globally_sparse(
    split_tracklets, MAX_LINK_GAP_BY_CLASS, MIN_LINK_SCORE, MOTION_WEIGHT, MOTION_NORM_M, MOTION_NORM_PX,
    velocity_extrapolation_cap_frames=VELOCITY_EXTRAPOLATION_CAP_FRAMES,
    classes_to_skip=LINK_SKIP, debug=DEBUG_LINKING,
    min_margin=LINK_MIN_MARGIN if STRICT_NO_SWITCH else 0.0
)
print(f"\nTotal accepted links: {len(accepted_links)}")
if SHOT_ID is not None:
    _xs = sum(1 for i, j in accepted_links.items()
              if SHOT_ID[split_tracklets[i]["frames"][-1]] != SHOT_ID[split_tracklets[j]["frames"][0]])
    print(f"  of which across a camera cut (position-continuous): {_xs}")


## Cell 9b -- Force goalkeepers into 2 IDs and referees into 3 IDs
Each GK / referee tracklet gets a **slot** from its pitch position:
* GK: left half -> `GK_left` (id 1), right half -> `GK_right` (id 2). First half only, so no side swap.
* Referee: mostly within `AR_BAND_M` of touchline y=0 -> `AR_y0` (id 4), of y=68 -> `AR_y68` (id 5), else `REF_main` (id 3).

Gates: cosine to the slot prototype >= `OFFICIAL_MIN_SIM` (rejects players mis-detected as GK/ref), and for the
assistants, the half they patrol. Then slots are filled most-confident first; a tracklet that overlaps its slot by more
than `SLOT_OVERLAP_TOL` frames is a duplicate (4th official, mis-detection, second box) and is dropped.
Read the diagnostics before trusting it: pitch-position ranges, and how often appearance disagrees with the side.

In [ ]:
from sklearn.cluster import KMeans

forced_slot_of_tracklet = {}          # tracklet idx -> slot name
dropped_official_tracklets = set()    # GK/ref tracklets that fit no slot
drop_reasons = Counter()
SLOT_ID = {**GK_SLOT_IDS, **REF_SLOT_IDS}
SLOT_CLASS = {**{s: "goalkeeper" for s in GK_SLOT_IDS}, **{s: "referee" for s in REF_SLOT_IDS}}


def sample_pitch_positions(tl, stride=POS_SAMPLE_STRIDE):
    pts = []
    for f in tl["frames"][::stride]:
        p = pitch_position(f, tl["bbox_by_frame"][f], homography_cache)
        if (p is not None and np.all(np.isfinite(p))
                and -15 < p[0] < PITCH_LENGTH + 15 and -15 < p[1] < PITCH_WIDTH + 15):
            pts.append(p)
    return np.array(pts) if len(pts) >= 3 else None


def slot_proto(idxs):
    sums = [split_tracklets[i]["emb_sum"] for i in idxs if split_tracklets[i]["emb_sum"] is not None]
    return _norm(np.sum(sums, axis=0)) if sums else None


def sim_to(i, proto):
    p = split_tracklets[i]["proto"]
    return None if (p is None or proto is None) else float(np.dot(p, proto))


def resolve_slots(cands, slot_names):
    """Most confident / longest first. A tracklet takes its first preferred slot whose occupancy overlap is
    <= SLOT_OVERLAP_TOL frames (those frames are trimmed off it); otherwise it is dropped."""
    occ = {s: np.zeros(n_frames + 1, dtype=bool) for s in slot_names}
    assigned, trims, rejected = {}, {}, []
    for cnd in sorted(cands, key=lambda d: (-d["conf"], -len(split_tracklets[d["idx"]]["frames"]))):
        fr = np.asarray(split_tracklets[cnd["idx"]]["frames"])
        for s in cnd["pref"]:
            ov = occ[s][fr]
            if ov.sum() <= SLOT_OVERLAP_TOL:
                keep = fr[~ov]
                occ[s][keep] = True
                assigned[cnd["idx"]] = s
                if ov.any():
                    trims[cnd["idx"]] = keep.tolist()
                break
        else:
            rejected.append(cnd["idx"])
    return assigned, trims, rejected


if FORCE_OFFICIALS:
    half = PITCH_LENGTH / 2
    # ------------------------------------------------------------------ goalkeepers
    gk_idx = [i for i, tl in enumerate(split_tracklets) if tl["class"] == "goalkeeper"]
    gk_side = {}
    medx = []
    for i in gk_idx:
        pts = sample_pitch_positions(split_tracklets[i])
        if pts is None:
            gk_side[i] = (None, 0.0)
            continue
        lf = float(np.mean(pts[:, 0] < half))
        medx.append(float(np.median(pts[:, 0])))
        gk_side[i] = ("GK_left" if lf >= 0.5 else "GK_right", abs(lf - 0.5) * 2)
    if medx:
        print(f"GK tracklets: {len(gk_idx)} | median pitch-x quantiles (5/25/50/75/95%): "
              f"{np.round(np.quantile(medx, [.05, .25, .5, .75, .95]), 1)}")

    if GK_ASSIGN_MODE == "appearance":
        ii = [i for i in gk_idx if split_tracklets[i]["proto"] is not None]
        km = KMeans(2, n_init=10, random_state=0).fit(
            np.stack([split_tracklets[i]["proto"] for i in ii]),
            sample_weight=[split_tracklets[i]["n_emb"] for i in ii])
        cl_name = {}
        for k in (0, 1):
            votes = Counter(gk_side[i][0] for i, l in zip(ii, km.labels_) if l == k and gk_side[i][0])
            cl_name[k] = votes.most_common(1)[0][0] if votes else None
        if cl_name[0] == cl_name[1] or None in cl_name.values():
            cl_name = {0: "GK_left", 1: "GK_right"}
        cents = np.stack([_norm(c) for c in km.cluster_centers_])
        for i, l in zip(ii, km.labels_):
            s = cents @ split_tracklets[i]["proto"]
            gk_side[i] = (cl_name[l], float(abs(s[0] - s[1])))
        print(f"GK_ASSIGN_MODE=appearance: cluster -> slot {cl_name}")

    seed = {s: [i for i in gk_idx if gk_side[i][0] == s and gk_side[i][1] >= 0.8
                and len(split_tracklets[i]["frames"]) >= 100] for s in GK_SLOT_IDS}
    gk_proto = {s: slot_proto(seed[s]) for s in GK_SLOT_IDS}
    if all(p is not None for p in gk_proto.values()):
        print(f"GK prototype cosine left vs right: {float(np.dot(gk_proto['GK_left'], gk_proto['GK_right'])):.3f} "
              f"(low = kits well separated)")
    disagree = [i for i in gk_idx if gk_side[i][0] and all(p is not None for p in gk_proto.values())
                and split_tracklets[i]["proto"] is not None
                and sim_to(i, gk_proto[gk_side[i][0]]) < sim_to(i, gk_proto[
                    "GK_right" if gk_side[i][0] == "GK_left" else "GK_left"])]
    print(f"GK tracklets where appearance prefers the OTHER side: {len(disagree)} "
          f"({sum(len(split_tracklets[i]['frames']) for i in disagree)} frames) "
          f"-> if large, homography sides are unreliable: set GK_ASSIGN_MODE='appearance'")

    gk_cands = []
    for i in gk_idx:
        side, conf = gk_side[i]
        if side is None:
            sims = {s: sim_to(i, gk_proto[s]) for s in GK_SLOT_IDS}
            if any(v is None for v in sims.values()):
                dropped_official_tracklets.add(i); drop_reasons["gk_no_position_no_embedding"] += 1; continue
            (s1, v1), (s2, v2) = sorted(sims.items(), key=lambda kv: -kv[1])
            if v1 - v2 < 0.05:
                dropped_official_tracklets.add(i); drop_reasons["gk_no_position_ambiguous_look"] += 1; continue
            side, conf = s1, 0.0
        sim = sim_to(i, gk_proto[side])
        if sim is not None and sim < OFFICIAL_MIN_SIM:
            dropped_official_tracklets.add(i); drop_reasons["gk_appearance_gate"] += 1; continue
        gk_cands.append({"idx": i, "pref": [side], "conf": conf})

    # ------------------------------------------------------------------ referees
    ref_idx = [i for i, tl in enumerate(split_tracklets) if tl["class"] == "referee"]
    ref_slot, ref_medx, ymed = {}, {}, []
    for i in ref_idx:
        pts = sample_pitch_positions(split_tracklets[i])
        if pts is None:
            ref_slot[i] = ("REF_main", 0.0)      # unknown position: only placed if the main slot is free
            continue
        fa = float(np.mean(pts[:, 1] < AR_BAND_M))
        fb = float(np.mean(pts[:, 1] > PITCH_WIDTH - AR_BAND_M))
        ref_medx[i] = float(np.median(pts[:, 0]))
        ymed.append(float(np.median(pts[:, 1])))
        if fa >= AR_MIN_FRAC:
            ref_slot[i] = ("AR_y0", fa)
        elif fb >= AR_MIN_FRAC:
            ref_slot[i] = ("AR_y68", fb)
        else:
            ref_slot[i] = ("REF_main", 1.0 - fa - fb)
    if ymed:
        print(f"\nReferee tracklets: {len(ref_idx)} | median pitch-y quantiles (5/25/50/75/95%): "
              f"{np.round(np.quantile(ymed, [.05, .25, .5, .75, .95]), 1)}")
    print("Referee slot proposals:", dict(Counter(v[0] for v in ref_slot.values())))

    ref_proto = slot_proto([i for i in ref_idx if ref_slot[i][1] >= 0.8 and len(split_tracklets[i]["frames"]) >= 100])
    ref_sims = [s for s in (sim_to(i, ref_proto) for i in ref_idx) if s is not None]
    if ref_sims:
        print(f"Referee cosine-to-prototype quantiles (5/25/50%): {np.round(np.quantile(ref_sims, [.05, .25, .5]), 3)} "
              f"(gate = {OFFICIAL_MIN_SIM})")

    ar_half = {}
    for s in ("AR_y0", "AR_y68"):
        xs = [(ref_medx[i], len(split_tracklets[i]["frames"])) for i in ref_idx
              if ref_slot[i][0] == s and ref_slot[i][1] >= 0.8 and i in ref_medx]
        if xs:
            x = np.repeat([a for a, _ in xs], [max(1, w // 25) for _, w in xs])
            ar_half[s] = "left" if np.median(x) < half else "right"
    print(f"Assistant patrol halves: {ar_half}")

    ref_cands = []
    for i in ref_idx:
        s, conf = ref_slot[i]
        sim = sim_to(i, ref_proto)
        if sim is not None and sim < OFFICIAL_MIN_SIM:
            dropped_official_tracklets.add(i); drop_reasons["ref_appearance_gate"] += 1; continue
        if AR_ENFORCE_HALF and s in ar_half and i in ref_medx:
            x = ref_medx[i]
            wrong = (x > half + AR_HALF_MARGIN_M) if ar_half[s] == "left" else (x < half - AR_HALF_MARGIN_M)
            if wrong:
                dropped_official_tracklets.add(i); drop_reasons["ar_wrong_half"] += 1; continue
        ref_cands.append({"idx": i, "pref": [s], "conf": conf})

    # ------------------------------------------------------------------ fill slots (one box per slot per frame)
    for cands, names in ((gk_cands, list(GK_SLOT_IDS)), (ref_cands, list(REF_SLOT_IDS))):
        assigned, trims, rejected = resolve_slots(cands, names)
        forced_slot_of_tracklet.update(assigned)
        for i in rejected:
            dropped_official_tracklets.add(i); drop_reasons["slot_occupied"] += 1
        for i, keep in trims.items():
            cls = split_tracklets[i]["class"]
            split_tracklets[i] = slice_tracklet(split_tracklets[i], keep)
            split_tracklets[i]["class"] = cls

    print(f"\n{'slot':<10} {'id':>3} {'tracklets':>9} {'frames':>8} {'first':>7} {'last':>7}")
    for s, sid in SLOT_ID.items():
        mem = [i for i, v in forced_slot_of_tracklet.items() if v == s]
        fr = [f for i in mem for f in split_tracklets[i]["frames"]]
        print(f"{s:<10} {sid:>3} {len(mem):>9} {len(fr):>8} {min(fr) if fr else '-':>7} {max(fr) if fr else '-':>7}")
    dropped_frames = sum(len(split_tracklets[i]["frames"]) for i in dropped_official_tracklets)
    print(f"\nDropped GK/ref tracklets: {len(dropped_official_tracklets)} ({dropped_frames} frames) | reasons: {dict(drop_reasons)}")
else:
    print("FORCE_OFFICIALS=False: goalkeepers/referees were linked like players in Cell 9.")


## Cell 10 -- Union links into identities (forced slots get fixed IDs 1-5, players start at 6)

In [ ]:
uf = UnionFind(len(split_tracklets))
for i, j in accepted_links.items():
    uf.union(i, j)

chain_members = defaultdict(list)
for idx in range(len(split_tracklets)):
    if idx in forced_slot_of_tracklet or idx in dropped_official_tracklets:
        continue
    chain_members[uf.find(idx)].append(idx)

canonical_id_of_tracklet = {}
canonical_class = {}
for idx, s in forced_slot_of_tracklet.items():
    canonical_id_of_tracklet[idx] = SLOT_ID[s]
    canonical_class[SLOT_ID[s]] = SLOT_CLASS[s]
forced_ids = set(SLOT_ID.values()) if FORCE_OFFICIALS else set()

next_id = FIRST_PLAYER_ID if FORCE_OFFICIALS else 1
for root, members in chain_members.items():
    for m in members:
        canonical_id_of_tracklet[m] = next_id
    canonical_class[next_id] = split_tracklets[members[0]]["class"]
    next_id += 1

chain_lengths = [len(members) for members in chain_members.values()]
print(f"Chained identities (non-forced): {len(chain_members)} | forced IDs: {sorted(forced_ids)}")
print(f"Chain length distribution: 1 tracklet={sum(1 for c in chain_lengths if c == 1)}, "
      f"2={sum(1 for c in chain_lengths if c == 2)}, 3+={sum(1 for c in chain_lengths if c >= 3)}")
print("Identity counts by class:", dict(Counter(canonical_class.values())))


## Cell 10b -- Player identity-level merge (minimum player IDs, not forced)
Stage-1 chaining only links tail->head within `MAX_LINK_GAP_BY_CLASS["player"]` using ~2 s of appearance. Here
whole identities are merged, best pair first, using the prototype of **all** their embeddings, with no gap limit.
Hard constraints (a merge that breaks one is never made):
* co-visible for more than `MERGE_OVERLAP_TOL` frames -> different people;
* different team (2-means on identity prototypes; check the co-visibility diagnostic before trusting it);
* any time boundary between the two identities that needs more than `MAX_SPEED_MPS` to cover.

The sweep prints the resulting player-ID count per threshold; pick `PLAYER_MERGE_SIM` at the knee, then confirm in
Cell 15 that `drift_tracks@t` and `class_flip_tracks` did not rise (a rise = wrong merges).

In [ ]:
if not PLAYER_MERGE:
    print("PLAYER_MERGE=False: no identity-level merge (zero-switch mode). Player IDs = stage-1 chains.")
    print("Identity counts by class:", dict(Counter(canonical_class.values())))
else:
    from scipy.sparse import csr_matrix

    player_ids = sorted(cid for cid, c in canonical_class.items() if cid not in forced_ids and c == "player")
    members_of = defaultdict(list)
    for idx, cid in canonical_id_of_tracklet.items():
        members_of[cid].append(idx)

    n_pid = len(player_ids)
    E0 = np.zeros((n_pid, 512), dtype=np.float32)
    n_emb0 = np.zeros(n_pid, dtype=np.int64)
    nframes0 = np.zeros(n_pid, dtype=np.int64)
    segs0, rows, cols = [], [], []
    for r, cid in enumerate(player_ids):
        segs = []
        for m in members_of[cid]:
            tl = split_tracklets[m]
            if tl["emb_sum"] is not None:
                E0[r] += tl["emb_sum"]; n_emb0[r] += tl["n_emb"]
            segs.append((tl["frames"][0], tl["frames"][-1], tl["head_anchor"], tl["tail_anchor"]))
        segs0.append(segs)
        fr = np.unique(np.concatenate([np.asarray(split_tracklets[m]["frames"]) for m in members_of[cid]]))
        nframes0[r] = len(fr)
        rows.append(np.full(len(fr), r)); cols.append(fr)
    Occ = csr_matrix((np.ones(int(nframes0.sum()), dtype=np.float32), (np.concatenate(rows), np.concatenate(cols))),
                     shape=(n_pid, n_frames + 1))
    OV0 = np.rint((Occ @ Occ.T).toarray()).astype(np.int64)
    np.fill_diagonal(OV0, 0)
    has_emb = n_emb0 > 0
    P0 = np.stack([_norm(e) if h else e for e, h in zip(E0, has_emb)]).astype(np.float32)
    print(f"Player identities entering the merge: {n_pid} (without any embedding: {int((~has_emb).sum())})")

    # ---- team split (2-means on prototypes of identities with enough frames) ----
    team = np.full(n_pid, -1, dtype=np.int64)
    if USE_TEAM_CONSTRAINT:
        fit = np.where(has_emb & (nframes0 >= TEAM_FIT_MIN_FRAMES))[0]
        km_team = KMeans(2, n_init=10, random_state=0).fit(P0[fit], sample_weight=nframes0[fit])
        team[has_emb] = km_team.predict(P0[has_emb])
        c0, c1 = (_norm(c) for c in km_team.cluster_centers_)
        covis = [np.asarray(Occ[team == t].sum(axis=0)).ravel() for t in (0, 1)]
        print(f"Team split: sizes {dict(Counter(team.tolist()))}, centroid cosine {float(np.dot(c0, c1)):.3f}")
        print(f"Max co-visible identities per team in one frame: {[int(c.max()) for c in covis]} "
              f"(99th pct {[int(np.quantile(c[c > 0], .99)) for c in covis]}) -- >10 outfield means the split is NOT by team; "
              f"set USE_TEAM_CONSTRAINT=False")


    def segments_feasible(sa, sb):
        segs = sorted(sa + sb, key=lambda s: s[0])
        return all(reachable(p[3], q[2]) for p, q in zip(segs, segs[1:]))


    def run_player_merge(thresh):
        P, E, OV = P0.copy(), E0.copy(), OV0.copy()
        tm = team.copy()
        segs = [list(s) for s in segs0]
        groups = [[r] for r in range(n_pid)]
        alive = np.ones(n_pid, dtype=bool)

        def valid_row(a):
            v = alive & has_emb & (OV[a] <= MERGE_OVERLAP_TOL)
            if USE_TEAM_CONSTRAINT and tm[a] >= 0:
                v &= (tm == tm[a]) | (tm < 0)
            v[a] = False
            return v

        M = np.full((n_pid, n_pid), -np.inf, dtype=np.float32)
        S = P @ P.T
        for a in np.where(has_emb)[0]:
            v = valid_row(a)
            M[a, v] = S[a, v]
        n_merge = n_infeasible = 0
        sims = []
        while True:
            k = int(np.argmax(M)); a, b = divmod(k, n_pid); s = float(M[a, b])
            if not np.isfinite(s) or s < thresh:
                break
            if not segments_feasible(segs[a], segs[b]):
                M[a, b] = M[b, a] = -np.inf; n_infeasible += 1
                continue
            segs[a] += segs[b]; groups[a] += groups[b]
            E[a] += E[b]; P[a] = _norm(E[a])
            if tm[a] < 0: tm[a] = tm[b]
            OV[a] += OV[b]; OV[:, a] = OV[a]; OV[a, a] = 0
            alive[b] = False
            M[b, :] = -np.inf; M[:, b] = -np.inf
            v = valid_row(a)
            row = np.full(n_pid, -np.inf, dtype=np.float32); row[v] = P[v] @ P[a]
            M[a, :] = row; M[:, a] = row
            n_merge += 1; sims.append(s)
        return [groups[r] for r in range(n_pid) if alive[r]], n_merge, n_infeasible, sims


    if PLAYER_MERGE and n_pid:
        print(f"\n{'thresh':>6} {'player_ids':>10} {'ids>=10s':>8} {'merges':>7} {'infeasible':>10} {'min_sim':>7}")
        for th in PLAYER_MERGE_SWEEP:
            g, nm, ni, sims = run_player_merge(th)
            long_ = sum(1 for grp in g if nframes0[grp].sum() >= 10 * FPS)
            print(f"{th:>6.2f} {len(g):>10} {long_:>8} {nm:>7} {ni:>10} {min(sims) if sims else float('nan'):>7.3f}")

        groups, nm, ni, _ = run_player_merge(PLAYER_MERGE_SIM)
        for c in player_ids:
            canonical_class.pop(c, None)
        nid = FIRST_PLAYER_ID if FORCE_OFFICIALS else (max(canonical_class) + 1 if canonical_class else 1)
        for grp in sorted(groups, key=lambda g: -nframes0[g].sum()):
            for r in grp:
                for m in members_of[player_ids[r]]:
                    canonical_id_of_tracklet[m] = nid
            canonical_class[nid] = "player"
            nid += 1
        print(f"\nApplied PLAYER_MERGE_SIM={PLAYER_MERGE_SIM}: {n_pid} -> {len(groups)} player identities "
              f"({nm} merges, {ni} blocked by the feasibility gate)")
    print("Identity counts by class:", dict(Counter(canonical_class.values())))


## Cell 11 -- Rebuild the final per-frame tracking cache with global IDs
Same format as before (`{frame: {"ball": ..., "tracks": [...]}}`), so downstream stages work unchanged.
Each track entry also keeps `raw_track_id` (the Deep-EIoU id) for auditing.

In [ ]:
final_tracking_cache = defaultdict(lambda: {"ball": None, "tracks": []})

for idx, tl in enumerate(split_tracklets):
    cid = canonical_id_of_tracklet.get(idx)
    if cid is None:                      # dropped GK/ref tracklet
        continue
    for f in tl["frames"]:
        final_tracking_cache[f]["tracks"].append({
            "track_id": cid,
            "bbox": tl["bbox_by_frame"][f],
            "conf": tl["conf_by_frame"][f],
            "class": tl["class_by_frame"][f],
            "raw_track_id": tl["raw_track_id"],
        })

# v2: one box per (frame, ID) -- keep the most confident (handoff overlaps allowed by the tolerances)
n_dup = 0
for f, data in final_tracking_cache.items():
    best = {}
    for t in data["tracks"]:
        b = best.get(t["track_id"])
        if b is None or t["conf"] > b["conf"]:
            best[t["track_id"]] = t
    n_dup += len(data["tracks"]) - len(best)
    data["tracks"] = list(best.values())
print(f"Duplicate (frame, ID) boxes removed: {n_dup}")

for f in range(n_frames):
    ball_det = next((d for d in detection_cache.get(f, []) if d["class"] == "ball"), None)
    final_tracking_cache.setdefault(f, {"ball": ball_det, "tracks": []})
    final_tracking_cache[f]["ball"] = ball_det

final_tracking_cache = dict(final_tracking_cache)
print(f"Rebuilt tracking cache: {len(final_tracking_cache)} frames.")

## Cell 12b -- Sustained class switches on the final player IDs (report only by default)
Forced IDs are skipped. With `SPLIT_FINAL_CLASS_SWITCH=False` the switches are only listed, because GK/ref tracklets no longer enter player chains and splitting would just add player IDs.

In [ ]:
frames_by_id = defaultdict(list)
for f, data in final_tracking_cache.items():
    for t in data["tracks"]:
        frames_by_id[t["track_id"]].append((f, t["class"]))

next_new_id = max(frames_by_id) + 1 if frames_by_id else 1
reassign = {}          # (frame, old_id) -> new_id
split_log = []
flagged = []
for tid, fc in frames_by_id.items():
    if tid in forced_ids:
        continue
    fc.sort()
    frames = [f for f, _ in fc]
    cuts = class_switch_cut_frames(frames, [c for _, c in fc], CLASS_SWITCH_SMOOTH_FRAMES, CLASS_SWITCH_MIN_RUN)
    if not cuts:
        continue
    flagged.append((tid, cuts))
    if not SPLIT_FINAL_CLASS_SWITCH:
        continue
    bounds = cuts + [frames[-1] + 1]
    for k, (a, b) in enumerate(zip(bounds[:-1], bounds[1:])):
        seg = [(f, c) for f, c in fc if a <= f < b]
        nid = next_new_id; next_new_id += 1
        for f, _ in seg:
            reassign[(f, tid)] = nid
        majority = Counter(c for _, c in seg).most_common(1)[0][0]
        split_log.append((tid, a, nid, majority, len(seg)))

for f, data in final_tracking_cache.items():
    for t in data["tracks"]:
        nid = reassign.get((f, t["track_id"]))
        if nid is not None:
            t["track_id"] = nid

for tid, cuts in flagged[:30]:
    print(f"  [flag] id {tid}: sustained class switch at frames {cuts[:5]}")
for s in split_log:
    canonical_class[s[2]] = "player"
print(f"IDs with a sustained class switch: {len({s[0] for s in split_log})} -> {len(split_log)} new segments")
for old_tid, start, nid, cls, n in split_log[:30]:
    print(f"  id {old_tid}: from frame {start} -> new id {nid} ({cls}, {n} frames)")

## Cell 12 -- Ghost-track removal + class locking (forced IDs keep their slot class, all others = player)

In [ ]:
id_frame_counts = defaultdict(int)
for data in final_tracking_cache.values():
    for t in data["tracks"]:
        id_frame_counts[t["track_id"]] += 1

ghost_ids = {tid for tid, count in id_frame_counts.items() if count < MIN_TRACK_LENGTH and tid not in forced_ids}
_rows_total = sum(id_frame_counts.values())
_rows_ghost = sum(id_frame_counts[t] for t in ghost_ids)
print(f"Ghost tracks dropped (< {MIN_TRACK_LENGTH} frames): {len(ghost_ids)} "
      f"({_rows_ghost} rows = {_rows_ghost / max(_rows_total, 1):.1%} of all rows; lower MIN_TRACK_LENGTH to keep them)")

for data in final_tracking_cache.values():
    data["tracks"] = [t for t in data["tracks"] if t["track_id"] not in ghost_ids]

class_votes = defaultdict(lambda: defaultdict(int))
for data in final_tracking_cache.values():
    for t in data["tracks"]:
        class_votes[t["track_id"]][t["class"]] += 1

locked_class_by_id = {}
for tid, votes in class_votes.items():
    if tid in forced_ids:
        locked_class_by_id[tid] = canonical_class[tid]
        continue
    if FORCE_OFFICIALS:                  # every GK/referee is in a forced slot -> everything else is a player
        locked_class_by_id[tid] = "player"
        continue
    total = sum(votes.values())
    ref_votes = votes.get("referee", 0)
    gk_votes = votes.get("goalkeeper", 0)
    ref_confirmed = ref_votes >= MIN_CONFIRM_FRAMES_ABS and (ref_votes / total) >= MIN_CONFIRM_RATIO
    gk_confirmed = gk_votes >= MIN_CONFIRM_FRAMES_ABS and (gk_votes / total) >= MIN_CONFIRM_RATIO
    if ref_confirmed:
        locked_class_by_id[tid] = "referee"
    elif gk_confirmed:
        locked_class_by_id[tid] = "goalkeeper"
    else:
        locked_class_by_id[tid] = max(votes, key=votes.get)

final_counts = defaultdict(int)
for cls in locked_class_by_id.values():
    final_counts[cls] += 1
print(f"\nFinal identity counts: {dict(final_counts)}")
if FORCE_OFFICIALS:
    for cls, expected in MAX_IDS_PER_CLASS_EXPECTED.items():
        got = final_counts.get(cls, 0)
        print(f"  [check] {cls}: {got} IDs (forced target {expected}) -> {'OK' if got == expected else 'CHECK: a slot is empty'}")


## Cell 13 -- Per-ID summary: detection counts, class votes, span
Counts are recomputed from the cache (so IDs created in Cell 12b are included), in one pass.

In [ ]:
frames_by_id = defaultdict(list)
for f, data in final_tracking_cache.items():
    for t in data["tracks"]:
        frames_by_id[t["track_id"]].append(f)

id_summary = []
for tid, frames_present in frames_by_id.items():
    frames_present.sort()
    count = len(frames_present)
    votes = class_votes[tid]
    total_votes = sum(votes.values())
    id_summary.append({
        "track_id": tid,
        "locked_class": locked_class_by_id[tid],
        "n_detections": count,
        "class_counts": dict(votes),
        "class_purity": round(votes.get(locked_class_by_id[tid], 0) / total_votes, 3) if total_votes else None,
        "first_frame": frames_present[0],
        "last_frame": frames_present[-1],
        "frame_span": frames_present[-1] - frames_present[0] + 1,
        "coverage_ratio": round(count / (frames_present[-1] - frames_present[0] + 1), 3),
    })

id_summary.sort(key=lambda r: (r["locked_class"], -r["frame_span"]))

print(f"{'id':>5} {'class':<10} {'n_det':>7} {'span':>8} {'cover':>6} {'purity':>7}  class_counts")
for r in id_summary:
    print(f"{r['track_id']:>5} {r['locked_class']:<10} {r['n_detections']:>7} "
          f"{r['frame_span']:>8} {r['coverage_ratio']:>6.2f} {r['class_purity']:>7}  {r['class_counts']}")

print(f"\ntotal IDs: {len(id_summary)}")
for cls in sorted(set(r["locked_class"] for r in id_summary)):
    cls_rows = [r for r in id_summary if r["locked_class"] == cls]
    print(f"  {cls}: {len(cls_rows)} IDs, "
          f"median coverage_ratio={sorted(r['coverage_ratio'] for r in cls_rows)[len(cls_rows) // 2]:.2f}")

low_coverage = sorted([r for r in id_summary if r["coverage_ratio"] < 0.05], key=lambda r: r["coverage_ratio"])
print(f"\nIDs with coverage_ratio < 0.05 (probably several stitched tracklets -- audit first): {len(low_coverage)}")
for r in low_coverage[:20]:
    print(f"  id={r['track_id']} class={r['locked_class']} n_det={r['n_detections']} span={r['frame_span']} cover={r['coverage_ratio']}")

pd.DataFrame(id_summary).to_csv(OUTPUT_SUMMARY_PATH, index=False)
print(f"\nSaved {OUTPUT_SUMMARY_PATH}")

## Cell 13b -- Class-purity check, all classes
Replaces the referee-only IoU re-match: the stored per-frame class is already the raw RF-DETR class.
A low purity (share of frames matching the locked class) means residual contamination worth checking.

In [ ]:
PURITY_WARN = 0.9
impure = sorted([r for r in id_summary if r["class_purity"] is not None and r["class_purity"] < PURITY_WARN],
                key=lambda r: r["class_purity"])
print(f"IDs with class purity < {PURITY_WARN}: {len(impure)} of {len(id_summary)}")
print(f"{'id':>5} {'locked':<10} {'purity':>7} {'n_det':>7}  class_counts")
for r in impure:
    print(f"{r['track_id']:>5} {r['locked_class']:<10} {r['class_purity']:>7} {r['n_detections']:>7}  {r['class_counts']}")

## Cell 14 -- Save final cache

In [ ]:
with open(OUTPUT_TRACKING_CACHE_PATH, "wb") as f:
    pickle.dump({"tracking_cache": final_tracking_cache, "locked_class_by_id": locked_class_by_id}, f)

print(f"Saved to {OUTPUT_TRACKING_CACHE_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_TRACKING_CACHE_PATH) / (1024 * 1024):.1f} MB")

## Cell 14b -- Save the inputs of notebook 2 (per-piece appearance, segments, team model)

In [ ]:
GROUPING_INPUTS_PATH = "/kaggle/working/grouping_inputs.pkl"

def _anchor(a):
    return None if a is None else (int(a[0]), np.asarray(a[1], dtype=np.float64).tolist())

_members = defaultdict(list)
for idx, cid in canonical_id_of_tracklet.items():
    _members[cid].append(idx)
id_feat = {}
for cid, cls in locked_class_by_id.items():
    if cls != "player":
        continue
    tls = [split_tracklets[m] for m in _members[cid]]
    es = [tl["emb_sum"] for tl in tls if tl.get("emb_sum") is not None]
    id_feat[cid] = {
        "emb_sum": np.sum(es, axis=0).astype(np.float32) if es else None,
        "n_emb": int(sum(tl.get("n_emb", 0) for tl in tls)),
        "segs": [(int(tl["frames"][0]), int(tl["frames"][-1]), _anchor(tl.get("head_anchor")), _anchor(tl.get("tail_anchor")))
                 for tl in tls],
    }
_params = {"FPS": FPS, "n_frames": int(n_frames), "MAX_SPEED_MPS": MAX_SPEED_MPS, "POS_SLACK_M": POS_SLACK_M,
           "MERGE_OVERLAP_TOL": MERGE_OVERLAP_TOL, "TEAM_MIN_MARGIN": TEAM_MIN_MARGIN, "FIRST_PLAYER_ID": FIRST_PLAYER_ID}
with open(GROUPING_INPUTS_PATH, "wb") as fh:
    pickle.dump({"id_feat": id_feat, "TEAM_CENTROIDS": TEAM_CENTROIDS, "params": _params}, fh)
print(f"Saved {GROUPING_INPUTS_PATH}: {len(id_feat)} player pieces "
      f"({os.path.getsize(GROUPING_INPUTS_PATH) / 1e6:.1f} MB) -> input of notebook 2")


## Cell 15 -- Audit: raw Deep-EIoU vs relinked, same label-free metrics
Same `evaluate_tracks` as the tracker comparison. For relinking, the metrics to watch are the
fragmentation ones (should drop) and `drift_tracks@t` / `class_flip_tracks` (must not rise: a rise
means links joined different people). Ghost IDs removed in Cell 12 are excluded from the relinked side.

In [ ]:
def evaluate_tracks(df, name, fps=FPS, n_input_dets=None, emb_npz=None, class_names=None,
                    max_gap=3, jump_ratio=0.75, flip_share=0.8, win_s=2.0,
                    drift_thresholds=(0.2, 0.3, 0.4), min_len_s=1.0):
    df = df.sort_values(['track_id', 'frame']).reset_index(drop=True)
    res = {'tracker': name}
    lens = df.groupby('track_id').size()
    res['ids_total']  = int(lens.size)
    res['ids_ge_1s']  = int((lens >= fps).sum())
    res['ids_ge_10s'] = int((lens >= 10 * fps).sum())
    res['median_len_s'] = round(float(lens.median()) / fps, 2)
    res['mean_len_s']   = round(float(lens.mean()) / fps, 2)
    res['rows'] = int(len(df))
    if n_input_dets:
        res['coverage'] = round(len(df) / n_input_dets, 3)

    # Duplicate IDs in the same frame
    res['dup_id_frames'] = int((df.groupby(['frame', 'track_id']).size() > 1).sum())

    # Abrupt jumps between near-consecutive observations of the same track (image space, scale-normalised)
    cx, cy = (df.x1 + df.x2).values / 2, (df.y1 + df.y2).values / 2
    h = (df.y2 - df.y1).values
    same = df.track_id.values[1:] == df.track_id.values[:-1]
    gap = df.frame.values[1:] - df.frame.values[:-1]
    step = np.hypot(np.diff(cx), np.diff(cy)) / np.maximum((h[1:] + h[:-1]) / 2, 1)
    jumps = same & (gap <= max_gap) & (step > jump_ratio)
    res['jump_violations']  = int(jumps.sum())
    res['tracks_with_jump'] = int(np.unique(df.track_id.values[1:][jumps]).size)

    # Class consistency and per-class ID counts (tracks >= min_len_s)
    long_ids = lens.index[lens >= min_len_s * fps]
    if 'cls' in df.columns and (df.cls >= 0).any():
        sub = df[df.track_id.isin(long_ids) & (df.cls >= 0)]
        share = sub.groupby('track_id').cls.agg(lambda s: s.value_counts(normalize=True).iloc[0])
        res['class_flip_tracks'] = int((share < flip_share).sum())
        major = sub.groupby('track_id').cls.agg(lambda s: s.value_counts().index[0])
        for c, n in major.value_counts().sort_index().items():
            res[f'ids_ge_1s_{(class_names or {}).get(int(c), int(c))}'] = int(n)

    # Appearance drift between adjacent windows of the same track
    if emb_npz is not None:
        z = np.load(emb_npz) if isinstance(emb_npz, (str, Path)) else emb_npz
        e = pd.DataFrame({'frame': z['frame'], 'track_id': z['track_id']})
        e['win'] = (e.frame // int(win_s * fps)).astype(np.int64)
        emb = z['emb'].astype(np.float32)
        e = e[e.track_id.isin(long_ids)]
        max_drift, adj_all = {}, []
        for tid, g in e.groupby('track_id'):
            wins = []
            for _, gw in g.groupby('win'):
                if len(gw) >= 3:
                    m = emb[gw.index.values].mean(0)
                    wins.append(m / max(np.linalg.norm(m), 1e-12))
            if len(wins) >= 2:
                W = np.stack(wins)
                d = 1 - (W[1:] * W[:-1]).sum(1)
                adj_all.append(d)
                max_drift[tid] = float(d.max())
        if adj_all:
            adj_all = np.concatenate(adj_all)
            res['drift_adjacent_median'] = round(float(np.median(adj_all)), 3)
            res['drift_adjacent_p95']    = round(float(np.quantile(adj_all, 0.95)), 3)
            md_ = np.array(list(max_drift.values()))
            for t in drift_thresholds:
                res[f'drift_tracks@{t}'] = int((md_ > t).sum())
    return res

def compare(*results):
    return pd.DataFrame(results).set_index('tracker').T


CLASS_NAME_TO_CLS = {v: k for k, v in CLS_ID_TO_NAME.items()}

rows = []
for f, data in final_tracking_cache.items():
    for t in data["tracks"]:
        x1, y1, x2, y2 = t["bbox"]
        rows.append((f, t["track_id"], x1, y1, x2, y2, t["conf"], CLASS_NAME_TO_CLS[t["class"]], t["raw_track_id"]))
relinked_df = pd.DataFrame(rows, columns=["frame", "track_id", "x1", "y1", "x2", "y2", "score", "cls", "raw_track_id"])

# Embeddings are keyed by (frame, Deep-EIoU id): re-key them to the relinked ids
key = pd.DataFrame({"frame": z["frame"], "raw_track_id": z["track_id"], "row": np.arange(len(z["frame"]))})
key = key.merge(relinked_df[["frame", "raw_track_id", "track_id"]], on=["frame", "raw_track_id"], how="inner")
relinked_embs = {"frame": key.frame.values, "track_id": key.track_id.values, "emb": z["emb"][key.row.values]}

raw_res = evaluate_tracks(tracks_df, "Deep-EIoU raw", emb_npz=DEEP_EIOU_EMBS_PATH, class_names=CLS_ID_TO_NAME)
rel_res = evaluate_tracks(relinked_df, "Relinked", emb_npz=relinked_embs, class_names=CLS_ID_TO_NAME)
audit = compare(raw_res, rel_res)
audit.to_csv(OUTPUT_AUDIT_PATH)
print(f"Saved {OUTPUT_AUDIT_PATH}")
audit

## Cell 16 -- Visual ID check: one contact sheet per final ID
One crop per stitched piece (raw id change or > 2 s gap) plus one every 30 s inside long pieces. All crops on a sheet must show the same person. The video is read sequentially (frame-accurate, ~5-10 min).

In [ ]:
# ============================================================================================
# Cell 16 -- Visual ID check: one contact sheet per final ID
# Each final ID is cut into SEGMENTS (a new segment starts whenever the Deep-EIoU raw id changes,
# the ID disappears for > SEG_GAP_FRAMES, or every SEG_MAX_FRAMES inside a long piece). One crop per segment = one crop per stitched piece,
# so every link/merge made by the relinking is visible. If all crops on a sheet show the same
# player (kit, number, hair, skin, boots), the ID is clean.
# The video is read SEQUENTIALLY (no seeking) so crops are frame-accurate -- ~5-10 min on Kaggle.
# ============================================================================================
import os, math, pickle, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

SHEET_DIR        = "/kaggle/working/id_sheets"
MAX_CROPS_PER_ID = 60        # segments above this are evenly subsampled (the header says so)
SEG_GAP_FRAMES   = 50        # 2 s without the ID -> new segment even if the raw id is the same
SEG_MAX_FRAMES   = 750       # also start a new segment every 30 s, so long raw tracks get several crops
CROP_H, CROP_W   = 160, 80
PAD_RATIO        = 0.10      # enlarge each box by 10% before cropping
COLS             = 12
IDS_TO_SHOW      = None      # None = every ID, or e.g. [6, 7, 12]
SHOW_INLINE      = True

VIDEO = globals().get("VIDEO_PATH", "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv")
FPS_  = globals().get("FPS", 25)

# ---- 1. load the final cache (from this session, else from disk) ----
try:
    cache, locked = final_tracking_cache, locked_class_by_id
except NameError:
    with open(OUTPUT_TRACKING_CACHE_PATH, "rb") as fh:
        _d = pickle.load(fh)
    cache, locked = _d["tracking_cache"], _d["locked_class_by_id"]

obs = defaultdict(list)                       # tid -> [(frame, raw_id, bbox)]
for f, data in cache.items():
    for t in data["tracks"]:
        obs[t["track_id"]].append((f, t.get("raw_track_id"), t["bbox"]))

ids = sorted(obs) if IDS_TO_SHOW is None else [i for i in IDS_TO_SHOW if i in obs]
if SHOW_INLINE and len(ids) > 60:
    print(f"{len(ids)} IDs: too many to show inline -> saved to disk only (set IDS_TO_SHOW to view some here)")
    SHOW_INLINE = False
order = {"goalkeeper": 0, "referee": 1, "player": 2}
ids.sort(key=lambda t: (order.get(locked.get(t), 3), -len(obs[t])))

# ---- 2. segments -> one representative frame each (largest box in the middle half) ----
wanted = defaultdict(list)                    # frame -> [(tid, k, bbox)]
seg_info = {}                                 # tid -> (list of (start, end, raw_id, rep_frame), n_segments_total)
for tid in ids:
    o = sorted(obs[tid], key=lambda x: x[0])
    segs, cur = [], [o[0]]
    for prev, x in zip(o, o[1:]):
        if x[1] != prev[1] or x[0] - prev[0] > SEG_GAP_FRAMES or x[0] - cur[0][0] >= SEG_MAX_FRAMES:
            segs.append(cur); cur = [x]
        else:
            cur.append(x)
    segs.append(cur)
    n_total = len(segs)
    if n_total > MAX_CROPS_PER_ID:
        keep = sorted(set(np.linspace(0, n_total - 1, MAX_CROPS_PER_ID).round().astype(int).tolist()))
        segs = [segs[k] for k in keep]
    info = []
    for k, s in enumerate(segs):
        mid = s[len(s) // 4: max(len(s) // 4 + 1, 3 * len(s) // 4)]
        rep = max(mid, key=lambda x: x[2][3] - x[2][1])
        wanted[rep[0]].append((tid, k, rep[2]))
        info.append((s[0][0], s[-1][0], s[0][1], rep[0]))
    seg_info[tid] = (info, n_total)

print(f"IDs: {len(ids)} | crops to extract: {sum(len(v[0]) for v in seg_info.values())} "
      f"from {len(wanted)} frames")


def letterbox(img):
    h, w = img.shape[:2]
    s = min(CROP_W / max(w, 1), CROP_H / max(h, 1))
    r = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    out = np.full((CROP_H, CROP_W, 3), 40, np.uint8)
    y0, x0 = (CROP_H - r.shape[0]) // 2, (CROP_W - r.shape[1]) // 2
    out[y0:y0 + r.shape[0], x0:x0 + r.shape[1]] = r
    return out


# ---- 3. read the video sequentially, crop the wanted frames ----
crops = {}
cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f"cannot open {VIDEO}"
last = max(wanted)
f = 0
with tqdm(total=last + 1, desc="Reading video", unit="frame", mininterval=1.0) as pbar:
    while f <= last:
        if f in wanted:
            ok, img = cap.read()
            if not ok:
                break
            H, W = img.shape[:2]
            for tid, k, (x1, y1, x2, y2) in wanted[f]:
                px, py = (x2 - x1) * PAD_RATIO, (y2 - y1) * PAD_RATIO
                a, b = int(max(0, x1 - px)), int(max(0, y1 - py))
                c, d = int(min(W, x2 + px)), int(min(H, y2 + py))
                if c > a and d > b:
                    crops[(tid, k)] = letterbox(img[b:d, a:c])
            pbar.set_postfix(crops=len(crops), refresh=False)
        else:
            if not cap.grab():
                break
        f += 1
        pbar.update(1)
cap.release()
print(f"Extracted {len(crops)} crops (stopped at frame {f})")

# ---- 4. build one sheet per ID ----
os.makedirs(SHEET_DIR, exist_ok=True)
LABEL_H, HEAD_H = 30, 34
font = cv2.FONT_HERSHEY_SIMPLEX
for tid in tqdm(ids, desc="Building sheets"):
    info, n_total = seg_info[tid]
    n = len(info)
    rows = math.ceil(n / COLS)
    sheet = np.full((HEAD_H + rows * (CROP_H + LABEL_H), COLS * CROP_W, 3), 255, np.uint8)
    head = (f"ID {tid} | {locked.get(tid, '?')} | {len(obs[tid])} det | {n_total} segments"
            + (f" (showing {n})" if n < n_total else ""))
    cv2.putText(sheet, head, (6, 23), font, 0.6, (0, 0, 0), 2, cv2.LINE_AA)
    for k, (s0, s1, raw, rep) in enumerate(info):
        r, cc = divmod(k, COLS)
        y, x = HEAD_H + r * (CROP_H + LABEL_H), cc * CROP_W
        crop = crops.get((tid, k))
        if crop is not None:
            sheet[y:y + CROP_H, x:x + CROP_W] = crop
        cv2.rectangle(sheet, (x, y), (x + CROP_W - 1, y + CROP_H - 1), (200, 200, 200), 1)
        t = rep / FPS_
        cv2.putText(sheet, f"{int(t // 60):02d}:{int(t % 60):02d} f{rep}", (x + 2, y + CROP_H + 12),
                    font, 0.3, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(sheet, f"raw {raw}", (x + 2, y + CROP_H + 25), font, 0.3, (90, 90, 90), 1, cv2.LINE_AA)
    cv2.imwrite(os.path.join(SHEET_DIR, f"id_{tid:04d}_{locked.get(tid, 'x')}.png"), sheet)
    if SHOW_INLINE:
        plt.figure(figsize=(COLS * 0.95, 0.4 + rows * 2.0))
        plt.imshow(cv2.cvtColor(sheet, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

shutil.make_archive(SHEET_DIR, "zip", SHEET_DIR)
print(f"Saved {len(ids)} sheets to {SHEET_DIR}/ and {SHEET_DIR}.zip")


## Cell 17 -- Trace an ID switch
Set `TRACE_ID` (sheet header) and `TRACE_FRAMES` (the `f...` label of a wrong crop). Lists every piece of the ID in time order and how it was joined to the previous one.

In [ ]:
# ============================================================================================
# Cell 17 -- Trace an ID switch: which step joined the wrong piece into this ID?
# Run in the same session as the relinking notebook (needs split_tracklets, uf, accepted_links, ...).
# Every tracklet of TRACE_ID is listed in time order with HOW it was joined to the previous one:
#   raw-internal   -> same Deep-EIoU raw id as the previous piece: the tracker itself may have switched
#   stage-1 link   -> Cell 9 greedy link (score shown)
#   10b merge      -> Cell 10b identity merge (cosine between the two stage-1 chains shown)
#   slot           -> forced GK/referee slot (Cell 9b)
# sim_to_id = cosine of the piece to the whole ID: the wrong player's pieces usually stand out low.
# ============================================================================================
TRACE_ID = 12                # final ID from the sheet header
TRACE_FRAMES = []            # optional: frame numbers of the wrong crops (the "f12345" labels)

members = sorted((i for i, c in canonical_id_of_tracklet.items() if c == TRACE_ID),
                 key=lambda i: split_tracklets[i]["frames"][0])
assert members, f"ID {TRACE_ID} not found (removed as ghost, or wrong id?)"
pred = {j: i for i, j in accepted_links.items()}

def _proto_of(idxs):
    s = [split_tracklets[i]["emb_sum"] for i in idxs if split_tracklets[i].get("emb_sum") is not None]
    return _norm(np.sum(s, axis=0)) if s else None

id_proto = _proto_of(members)
chain_proto = {}
for i in members:
    r = uf.find(i)
    if r not in chain_proto:
        chain_proto[r] = _proto_of([m for m in range(len(split_tracklets)) if uf.find(m) == r])

print(f"ID {TRACE_ID} ({locked_class_by_id.get(TRACE_ID, '?')}): {len(members)} tracklets, "
      f"{len({uf.find(i) for i in members})} stage-1 chains")
print(f"{'':2}{'tracklet':>8} {'raw':>6} {'start':>7} {'end':>7} {'len':>6} {'class':<10} "
      f"{'chain':>6} {'sim_to_id':>9}  joined to previous piece by")
prev = None
for i in members:
    tl = split_tracklets[i]
    fr = tl["frames"]
    p = tl.get("proto")
    sim = f"{float(np.dot(p, id_proto)):.3f}" if (p is not None and id_proto is not None) else "-"
    if prev is None:
        how = "(first piece)"
    elif i in forced_slot_of_tracklet:
        how = f"slot {forced_slot_of_tracklet[i]}"
    elif pred.get(i) == prev:
        sc = link_score(split_tracklets[prev], tl, MOTION_WEIGHT, MOTION_NORM_M, MOTION_NORM_PX,
                        MAX_LINK_GAP_BY_CLASS.get(tl["class"], 500), VELOCITY_EXTRAPOLATION_CAP_FRAMES)
        how = f"stage-1 link, score {sc:.3f}" if sc is not None else "stage-1 link"
        if split_tracklets[prev]["raw_track_id"] == tl["raw_track_id"]:
            how += " (Cell 8 had cut this raw track here)"
    elif uf.find(i) == uf.find(prev):
        how = "same stage-1 chain (pieces interleaved with another chain)"
    else:
        a, b = chain_proto.get(uf.find(prev)), chain_proto.get(uf.find(i))
        how = f"10b merge, chain cosine {float(np.dot(a, b)):.3f}" if (a is not None and b is not None) else "10b merge"
    hit = "->" if any(fr[0] <= f <= fr[-1] for f in TRACE_FRAMES) else "  "
    print(f"{hit}{i:>8} {tl['raw_track_id']:>6} {fr[0]:>7} {fr[-1]:>7} {len(fr):>6} {tl['class']:<10} "
          f"{uf.find(i):>6} {sim:>9}  {how}")
    prev = i
print("\n'->' = tracklet containing a TRACE_FRAMES frame. If that piece has the same raw id as its crop "
      "neighbours on the sheet, the switch is INSIDE a Deep-EIoU track (not a link or merge).")
